## Fuente de datos y selección de variables

Los datos utilizados en este proyecto proceden del **Bureau of Transportation Statistics (BTS)** de Estados Unidos, concretamente del conjunto de datos de puntualidad y rendimiento operativo de vuelos comerciales. https://www.transtats.bts.gov/DL_SelectFields.aspx?gnoyr_VQ=FGK&QO_fu146_anzr=b0-gvzr

Con el objetivo de garantizar que todos los periodos descargados presenten una estructura homogénea, se ha definido previamente un conjunto común de variables. Esta selección responde a tres necesidades principales del proyecto:

1. Disponer de información suficiente para realizar el análisis exploratorio de datos.
2. Conservar las variables necesarias para construir modelos explicativos y predictivos.
3. Mantener información adicional útil para la interpretación de resultados y la visualización posterior en Tableau.

La selección de una variable para el dataset maestro no implica necesariamente que dicha variable vaya a utilizarse como predictor en los modelos. Algunas variables contienen información que únicamente está disponible una vez iniciado o finalizado el vuelo, por lo que su utilización como predictor podría introducir **data leakage**.

### Variables seleccionadas

#### Información temporal

- `FL_DATE`: fecha correspondiente al vuelo.

Las siguientes variables temporales no se descargan directamente:

- `YEAR`
- `QUARTER`
- `MONTH`
- `DAY_OF_MONTH`
- `DAY_OF_WEEK`

Estas características serán generadas posteriormente a partir de `FL_DATE` durante la fase de ingeniería de características. De esta forma se evita almacenar información redundante y se mantiene una única fuente temporal de referencia.

---

#### Información de la aerolínea

- `MKT_UNIQUE_CARRIER`: código de la aerolínea comercial responsable del vuelo.
- `OP_UNIQUE_CARRIER`: código de la aerolínea que opera efectivamente el vuelo.
- `TAIL_NUM`: matrícula o identificador de la aeronave.
- `OP_CARRIER_FL_NUM`: número de vuelo asignado por la aerolínea operadora.

---

#### Aeropuerto de origen

- `ORIGIN_AIRPORT_ID`: identificador único del aeropuerto de origen.
- `ORIGIN`: código del aeropuerto de origen.
- `ORIGIN_CITY_NAME`: ciudad asociada al aeropuerto de origen.
- `ORIGIN_STATE_ABR`: abreviatura del estado correspondiente al aeropuerto de origen.

---

#### Aeropuerto de destino

- `DEST_AIRPORT_ID`: identificador único del aeropuerto de destino.
- `DEST`: código del aeropuerto de destino.
- `DEST_CITY_NAME`: ciudad asociada al aeropuerto de destino.
- `DEST_STATE_ABR`: abreviatura del estado correspondiente al aeropuerto de destino.

---

#### Información de salida

- `CRS_DEP_TIME`: hora programada de salida.
- `DEP_TIME`: hora real de salida.
- `DEP_DELAY`: diferencia en minutos entre la salida programada y la salida real. Puede contener valores negativos cuando el vuelo sale antes de la hora prevista.
- `DEP_DELAY_NEW`: minutos de retraso en salida, asignando cero a las salidas anticipadas.
- `DEP_DEL15`: indicador de retraso de salida igual o superior a 15 minutos.
- `DEP_TIME_BLK`: bloque horario correspondiente a la salida programada.
- `TAXI_OUT`: tiempo transcurrido entre la salida de la puerta y el despegue.

---

#### Información de llegada

- `TAXI_IN`: tiempo transcurrido entre el aterrizaje y la llegada a la puerta.
- `CRS_ARR_TIME`: hora programada de llegada.
- `ARR_TIME`: hora real de llegada.
- `ARR_DELAY`: diferencia en minutos entre la llegada programada y la llegada real. Puede presentar valores negativos en llegadas anticipadas.
- `ARR_DELAY_NEW`: minutos positivos de retraso en la llegada, asignando cero a las llegadas anticipadas.
- `ARR_DEL15`: indicador de retraso de llegada igual o superior a 15 minutos.
- `ARR_TIME_BLK`: bloque horario correspondiente a la llegada programada.

`ARR_DELAY_NEW` constituye una posible variable objetivo para los modelos de regresión, mientras que `ARR_DEL15` podrá utilizarse como variable objetivo en un eventual problema de clasificación.

---

#### Cancelaciones y desvíos

- `CANCELLED`: indicador de vuelo cancelado.
- `CANCELLATION_CODE`: código asociado al motivo de cancelación.
- `DIVERTED`: indicador de vuelo desviado.

Estas variables permitirán analizar de forma independiente los vuelos que no completaron el trayecto según la programación original y establecer las reglas necesarias para su inclusión o exclusión en cada modelo.

---

#### Características generales del vuelo

- `CRS_ELAPSED_TIME`: duración programada del vuelo.
- `ACTUAL_ELAPSED_TIME`: duración real del vuelo.
- `AIR_TIME`: tiempo efectivo de vuelo.
- `DISTANCE`: distancia entre los aeropuertos de origen y destino.
- `DISTANCE_GROUP`: categoría de distancia utilizada por BTS.

---

#### Causas de retraso

- `CARRIER_DELAY`: minutos de retraso atribuibles a la aerolínea.
- `WEATHER_DELAY`: minutos de retraso asociados a condiciones meteorológicas.
- `NAS_DELAY`: retraso atribuible al National Air System.
- `SECURITY_DELAY`: retraso asociado a cuestiones de seguridad.
- `LATE_AIRCRAFT_DELAY`: retraso provocado por la llegada tardía de la aeronave utilizada previamente.

Estas variables se conservarán principalmente para análisis exploratorio, interpretación de resultados y visualización. Al tratarse de información que se conoce después de producirse el retraso, no se utilizarán como predictores en un modelo destinado a realizar predicciones antes de la salida del vuelo.

---

### Número total de variables

El esquema maestro utilizado en este proyecto está compuesto por **40 variables originales**.

Este esquema deberá mantenerse constante en todos los archivos descargados para garantizar que los diferentes periodos puedan integrarse posteriormente sin generar inconsistencias estructurales.

---

## Tablas de referencia (`Lookup Tables`)

Además del dataset principal, se descargarán determinadas tablas de referencia proporcionadas por BTS.

Estas tablas permiten convertir códigos utilizados en el dataset principal en descripciones comprensibles y facilitan tanto el análisis como la interpretación de los resultados.

### Aerolíneas

Se utilizará el catálogo correspondiente a:

- `MKT_UNIQUE_CARRIER`
- `OP_UNIQUE_CARRIER`

El objetivo es transformar códigos de aerolínea en nombres descriptivos.

Ejemplo:

    AA → American Airlines
    DL → Delta Air Lines
    UA → United Airlines

Este enriquecimiento será especialmente útil durante el análisis exploratorio y en las visualizaciones desarrolladas en Tableau.

---

### Aeropuertos

Se descargará la tabla de referencia asociada a:

- `ORIGIN_AIRPORT_ID`
- `DEST_AIRPORT_ID`

Los identificadores de aeropuerto permiten mantener una referencia estable a lo largo del tiempo y pueden utilizarse para enriquecer los datos con información descriptiva adicional sobre cada aeropuerto.

Una misma tabla de referencia podrá utilizarse tanto para los aeropuertos de origen como para los de destino.

---

### Motivos de cancelación

Se descargará el catálogo correspondiente a:

- `CANCELLATION_CODE`

Esta tabla permitirá sustituir los códigos de cancelación por sus descripciones oficiales y facilitar el análisis de las diferentes causas de cancelación.

---

## Variables derivadas

Durante la fase de preparación e ingeniería de características se generarán variables temporales a partir de `FL_DATE`.

Entre ellas:

- Año.
- Trimestre.
- Mes.
- Día del mes.
- Día de la semana.
- Indicador de fin de semana.

Estas variables permitirán estudiar tendencias y patrones temporales sin necesidad de almacenar información redundante en los archivos originales.

---

## Consideración sobre Data Leakage

El dataset contiene variables generadas en diferentes momentos del ciclo operativo de un vuelo.

Para los modelos predictivos se distinguirá entre:

1. Información disponible antes de la salida del vuelo.
2. Información disponible después de la salida.
3. Información conocida únicamente al finalizar el vuelo.

Esta distinción es necesaria para evitar que los modelos utilicen información futura durante el entrenamiento.

Por ejemplo, variables como:

- `ARR_TIME`
- `ARR_DELAY`
- `ARR_DELAY_NEW`
- `CARRIER_DELAY`
- `WEATHER_DELAY`
- `NAS_DELAY`
- `SECURITY_DELAY`
- `LATE_AIRCRAFT_DELAY`

no podrán utilizarse como variables predictoras en un modelo diseñado para estimar el retraso antes de la salida, ya que contienen información que aún no existiría en el momento real de realizar la predicción.

Por tanto, el dataset maestro conservará toda la información necesaria para el análisis global del proyecto, mientras que cada modelo utilizará únicamente el subconjunto de variables compatible con el momento en el que se pretende realizar la predicción.

# 01. Ingesta y reconocimiento inicial de los datos

## Objetivo

El objetivo de este notebook es realizar una primera inspección de los archivos históricos de vuelos utilizados en el proyecto.

Los datos se encuentran distribuidos en múltiples carpetas, cada una de las cuales contiene un archivo CSV descargado del Bureau of Transportation Statistics (BTS).

Debido al volumen total del conjunto de datos, superior a 1 GB, no se realizará inicialmente una carga completa en memoria. En su lugar, se seguirá una estrategia de exploración progresiva que permita:

- Identificar automáticamente todos los archivos disponibles.
- Comprobar el número total de archivos.
- Analizar el tamaño individual y total de los datos.
- Validar que todos los archivos utilizan la misma estructura de columnas.
- Inspeccionar una muestra de los datos.
- Analizar los tipos de datos detectados inicialmente por Pandas.

Esta estrategia permite comprender la estructura de la fuente antes de definir las reglas de limpieza, normalización y transformación que se aplicarán posteriormente.

## 1. Importación de librerías

Para esta primera fase únicamente se requieren dos componentes:

- `pathlib.Path`: permite gestionar rutas y archivos de forma independiente del sistema operativo y evita concatenar manualmente cadenas de texto.
- `pandas`: se utilizará para construir el inventario de archivos y realizar las primeras inspecciones de los CSV.

En esta etapa se evita importar librerías que todavía no son necesarias, manteniendo el notebook sencillo y reduciendo dependencias innecesarias.


In [1]:
from pathlib import Path
import sys

import pandas as pd

# Parámetros generales de ejecución.
# 40 = columnas originales descargadas del BTS.
# 100_000 = equilibrio entre rendimiento y consumo de memoria.
EXPECTED_NUMBER_OF_COLUMNS = 40
CHUNK_SIZE = 100_000


## 2. Configuración del proyecto e importación de módulos reutilizables

El notebook no contiene la implementación interna de las funciones estables del pipeline.
Las rutas se centralizan en `config/paths.py` y la lógica reutilizable se encuentra en
`src/preprocessing.py` y `src/ingestion.py`.

Esta separación evita duplicación de código y garantiza que las futuras cargas utilicen
exactamente las mismas reglas de transformación.


In [2]:
# ============================================================
# CONFIGURACIÓN DE LA RAÍZ DEL PROYECTO
# ============================================================

# El notebook se encuentra dentro de /notebooks, mientras que los paquetes
# config/ y src/ viven en la raíz. Añadir PROJECT_ROOT a sys.path permite
# importarlos sin copiar funciones dentro del notebook.
PROJECT_ROOT = Path(r"G:\My Drive\MASTER Big Data\TFM")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Raíz del proyecto: {PROJECT_ROOT}")
print(f"Raíz disponible para imports: {str(PROJECT_ROOT) in sys.path}")


Raíz del proyecto: G:\My Drive\MASTER Big Data\TFM
Raíz disponible para imports: True


In [3]:
# Rutas centralizadas. Si cambia la ubicación física del proyecto,
# solamente debe modificarse config/paths.py.
from config.paths import (
    PROJECT_ROOT,
    FLIGHTS_DIR,
    PROCESSED_FLIGHTS_DIR,
    REJECTED_DIR,
)

# Esquema y reglas de transformación comunes a todo el histórico.
from src.preprocessing import (
    EXPECTED_RAW_COLUMNS,
    CATEGORICAL_COLUMNS,
    IDENTIFIER_COLUMNS,
    BINARY_COLUMNS,
    TIME_COLUMNS,
    CONTINUOUS_COLUMNS,
    ORDINAL_COLUMNS,
    FINAL_COLUMN_ORDER,
    normalize_chunk,
    clean_chunk,
)

# Funciones necesarias para la ejecución incremental.
from src.ingestion import (
    calculate_sha256,
    discover_csv_files,
    load_ingestion_manifest,
    detect_new_files,
)


## 3. Localización de los archivos CSV

Se utiliza `discover_csv_files()` para localizar recursivamente los CSV disponibles en la
capa `raw`. La lógica vive en `src/ingestion.py`, por lo que podrá reutilizarse durante
futuras actualizaciones sin duplicarla en otros notebooks.


## Mapa de ejecución del notebook

Este notebook contiene dos tipos de lógica:

**Construcción y validación histórica (bloques 3–19).** Documenta cómo se exploró,
normalizó, validó y generó por primera vez la capa histórica. Los análisis exhaustivos de
calidad no deben repetirse rutinariamente cada vez que aparezca un nuevo archivo.

**Actualización productiva (bloque 20).** Es el flujo que debe reutilizarse cuando BTS
publique nuevos periodos. Primero consulta el manifest y únicamente procesa las fuentes
pendientes.

Las funciones estables no se redefinen aquí: se importan desde `src/`.


In [4]:
# Inventario actual de fuentes. Esta llamada NO carga los datos:
# únicamente devuelve las rutas de los CSV presentes en data/raw/Flights.
csv_files = discover_csv_files(FLIGHTS_DIR)

print(f"Número total de archivos CSV encontrados: {len(csv_files)}")


Número total de archivos CSV encontrados: 53


## 4. Comprobación de los archivos encontrados

Antes de procesar los datos se realiza una inspección visual de algunos de los archivos detectados.

El objetivo es comprobar que la búsqueda recursiva ha localizado correctamente los CSV almacenados dentro de las diferentes carpetas y que las rutas obtenidas corresponden a los datos esperados.

Para evitar generar una salida innecesariamente extensa, únicamente se muestran los primeros cinco archivos.

In [5]:
for i, csv_file in enumerate(csv_files[:5], start=1):
    print(f"Archivo {i}")
    print(f"Carpeta : {csv_file.parent.name}")
    print(f"CSV     : {csv_file.name}")
    print()

Archivo 1
Carpeta : T_ONTIME_MARKETING_20260807_125204
CSV     : T_ONTIME_MARKETING.csv

Archivo 2
Carpeta : T_ONTIME_MARKETING_20260807_125336
CSV     : T_ONTIME_MARKETING.csv

Archivo 3
Carpeta : T_ONTIME_MARKETING_20260807_125558
CSV     : T_ONTIME_MARKETING.csv

Archivo 4
Carpeta : T_ONTIME_MARKETING_20260807_125711
CSV     : T_ONTIME_MARKETING.csv

Archivo 5
Carpeta : T_ONTIME_MARKETING_20260807_125821
CSV     : T_ONTIME_MARKETING.csv



## 5. Creación del inventario de archivos

Antes de cargar el contenido de los CSV se construye un inventario que resume las características físicas de las fuentes disponibles.

Para cada archivo se almacena:

- Carpeta de origen.
- Nombre del archivo.
- Tamaño en megabytes.
- Ruta completa.

Este inventario permite conocer el volumen real de información que se procesará y ayuda a definir posteriormente una estrategia de lectura eficiente.

In [6]:
inventory = []

for csv_file in csv_files:
    size_bytes = csv_file.stat().st_size

    inventory.append(
        {
            "folder": csv_file.parent.name,
            "file": csv_file.name,
            "size_mb": size_bytes / (1024 ** 2),
            "path": str(csv_file),
        }
    )

inventory_df = pd.DataFrame(inventory)

inventory_df.head()

,folder,file,size_mb,path
0,T_ONTIME_MARKETING_20260807_125204,T_ONTIME_MARKETING.csv,149.775033,G:\My Drive\MASTER Big Data\TFM\data\raw\Fligh...
1,T_ONTIME_MARKETING_20260807_125336,T_ONTIME_MARKETING.csv,145.976215,G:\My Drive\MASTER Big Data\TFM\data\raw\Fligh...
2,T_ONTIME_MARKETING_20260807_125558,T_ONTIME_MARKETING.csv,149.141684,G:\My Drive\MASTER Big Data\TFM\data\raw\Fligh...
3,T_ONTIME_MARKETING_20260807_125711,T_ONTIME_MARKETING.csv,125.152705,G:\My Drive\MASTER Big Data\TFM\data\raw\Fligh...
4,T_ONTIME_MARKETING_20260807_125821,T_ONTIME_MARKETING.csv,132.025734,G:\My Drive\MASTER Big Data\TFM\data\raw\Fligh...


## 6. Análisis del volumen de información

Se calculan estadísticas básicas sobre el tamaño de los archivos.

Estas métricas permiten conocer:

- Número total de archivos.
- Volumen total de datos.
- Tamaño medio de cada archivo.
- Archivo de menor tamaño.
- Archivo de mayor tamaño.

Este análisis es especialmente importante debido al gran volumen del dataset, ya que permite justificar la utilización posterior de técnicas de procesamiento por bloques (`chunks`) en lugar de cargar simultáneamente todos los datos en memoria.

In [7]:
print(f"Número de archivos : {len(inventory_df):,}")
print(
    f"Tamaño total       : "
    f"{inventory_df['size_mb'].sum():,.2f} MB"
)
print(
    f"Tamaño medio       : "
    f"{inventory_df['size_mb'].mean():,.2f} MB"
)
print(
    f"Archivo más pequeño: "
    f"{inventory_df['size_mb'].min():,.2f} MB"
)
print(
    f"Archivo más grande : "
    f"{inventory_df['size_mb'].max():,.2f} MB"
)

Número de archivos : 53
Tamaño total       : 7,229.09 MB
Tamaño medio       : 136.40 MB
Archivo más pequeño: 113.60 MB
Archivo más grande : 154.36 MB


## 7. Validación de la estructura de los archivos

Antes de combinar información procedente de diferentes archivos es necesario comprobar que todos contienen la misma estructura.

Para realizar esta validación no es necesario cargar los registros completos. Pandas permite leer únicamente la cabecera del CSV utilizando `nrows=0`.

De esta forma se pueden recuperar los nombres y el orden de las columnas con un consumo mínimo de memoria.

Posteriormente se agrupan los archivos en función de su esquema. Si todos los archivos presentan exactamente las mismas columnas y en el mismo orden, se obtendrá un único esquema.

La existencia de múltiples esquemas indicaría que los archivos presentan diferencias estructurales que deberán resolverse antes de la integración.

In [8]:
schemas = {}

for csv_file in csv_files:
    columns = tuple(
        pd.read_csv(
            csv_file,
            nrows=0
        ).columns
    )

    schemas.setdefault(
        columns,
        []
    ).append(csv_file)

print(
    f"Número de esquemas diferentes encontrados: "
    f"{len(schemas)}"
)

Número de esquemas diferentes encontrados: 1


## 8. Inspección de las variables disponibles

Una vez identificados los diferentes esquemas, se muestran las columnas que componen cada uno.

Esta comprobación permite verificar que las variables descargadas coinciden con las seleccionadas previamente para el proyecto y constituye una primera validación del diccionario de datos.

También permite identificar posibles diferencias de nomenclatura entre la documentación del BTS y los nombres reales incluidos en los archivos CSV.

In [9]:
for schema_number, (columns, files) in enumerate(
    schemas.items(),
    start=1
):
    print("=" * 70)
    print(f"ESQUEMA {schema_number}")
    print("=" * 70)

    print(f"Archivos asociados : {len(files)}")
    print(f"Número de columnas : {len(columns)}")

    print("\nVariables:")

    for i, column in enumerate(columns, start=1):
        print(f"{i:02d}. {column}")

    print()

ESQUEMA 1
Archivos asociados : 53
Número de columnas : 40

Variables:
01. FL_DATE
02. MKT_UNIQUE_CARRIER
03. OP_UNIQUE_CARRIER
04. TAIL_NUM
05. OP_CARRIER_FL_NUM
06. ORIGIN_AIRPORT_ID
07. ORIGIN
08. ORIGIN_CITY_NAME
09. ORIGIN_STATE_ABR
10. DEST_AIRPORT_ID
11. DEST
12. DEST_CITY_NAME
13. DEST_STATE_ABR
14. CRS_DEP_TIME
15. DEP_TIME
16. DEP_DELAY
17. DEP_DELAY_NEW
18. DEP_DEL15
19. DEP_TIME_BLK
20. TAXI_OUT
21. TAXI_IN
22. CRS_ARR_TIME
23. ARR_TIME
24. ARR_DELAY
25. ARR_DELAY_NEW
26. ARR_DEL15
27. ARR_TIME_BLK
28. CANCELLED
29. CANCELLATION_CODE
30. DIVERTED
31. CRS_ELAPSED_TIME
32. ACTUAL_ELAPSED_TIME
33. AIR_TIME
34. DISTANCE
35. DISTANCE_GROUP
36. CARRIER_DELAY
37. WEATHER_DELAY
38. NAS_DELAY
39. SECURITY_DELAY
40. LATE_AIRCRAFT_DELAY



## 9. Comprobación del número esperado de variables

El conjunto de datos fue descargado seleccionando previamente 40 variables relacionadas con información temporal, aerolíneas, aeropuertos, horarios, retrasos, cancelaciones, duración del vuelo y causas de retraso.

Se comprueba que el esquema obtenido contiene exactamente dicho número de columnas.

Esta validación permite detectar inmediatamente archivos incompletos o configuraciones de descarga diferentes.

In [10]:

for schema_number, columns in enumerate(
    schemas.keys(),
    start=1
):
    number_of_columns = len(columns)

    print(
        f"Esquema {schema_number}: "
        f"{number_of_columns} columnas"
    )

    if number_of_columns == EXPECTED_NUMBER_OF_COLUMNS:
        print("✓ Número de columnas correcto")
    else:
        print(
            "⚠ Número de columnas diferente "
            "al esperado"
        )

Esquema 1: 40 columnas
✓ Número de columnas correcto


## 10. Lectura de una muestra de los datos

Una vez validada la estructura de los archivos se realiza una primera lectura del contenido.

En esta fase no se carga el archivo completo. Se utilizan únicamente diez registros del primer CSV localizado.

El objetivo es observar:

- Formato de las variables.
- Representación de valores faltantes.
- Formato de las fechas.
- Formato de horarios.
- Variables categóricas.
- Variables numéricas.
- Indicadores binarios.

Esta muestra será utilizada únicamente para diseñar las reglas iniciales de normalización. Las conclusiones sobre calidad de datos no se realizarán todavía sobre estos diez registros, ya que no constituyen una muestra estadísticamente representativa del conjunto completo.

In [11]:
sample_file = csv_files[0]

sample_df = pd.read_csv(
    sample_file,
    nrows=10
)

print("Archivo utilizado:")
print(sample_file)

print(
    f"\nDimensiones de la muestra: "
    f"{sample_df.shape[0]} filas x "
    f"{sample_df.shape[1]} columnas"
)

sample_df

Archivo utilizado:
G:\My Drive\MASTER Big Data\TFM\data\raw\Flights\T_ONTIME_MARKETING_20260807_125204\T_ONTIME_MARKETING.csv

Dimensiones de la muestra: 10 filas x 40 columnas


,FL_DATE,MKT_UNIQUE_CARRIER,OP_UNIQUE_CARRIER,TAIL_NUM,OP_CARRIER_FL_NUM,ORIGIN_AIRPORT_ID,ORIGIN,ORIGIN_CITY_NAME,ORIGIN_STATE_ABR,DEST_AIRPORT_ID,...,CRS_ELAPSED_TIME,ACTUAL_ELAPSED_TIME,AIR_TIME,DISTANCE,DISTANCE_GROUP,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY
0,5/1/2026 12:00:00 AM,AA,AA,N101NN,234,14771,SFO,"San Francisco, CA",CA,12478,...,351.0,325.0,300.0,2586.0,11,NaN,NaN,NaN,NaN,NaN
1,5/1/2026 12:00:00 AM,AA,AA,N101NN,2453,12892,LAX,"Los Angeles, CA",CA,10721,...,334.0,324.0,293.0,2611.0,11,NaN,NaN,NaN,NaN,NaN
2,5/1/2026 12:00:00 AM,AA,AA,N101NN,302,12478,JFK,"New York, NY",NY,12892,...,384.0,350.0,325.0,2475.0,10,NaN,NaN,NaN,NaN,NaN
3,5/1/2026 12:00:00 AM,AA,AA,N102NN,255,12478,JFK,"New York, NY",NY,12892,...,375.0,385.0,337.0,2475.0,10,NaN,NaN,NaN,NaN,NaN
4,5/1/2026 12:00:00 AM,AA,AA,N102NN,4,12892,LAX,"Los Angeles, CA",CA,12478,...,332.0,320.0,286.0,2475.0,10,NaN,NaN,NaN,NaN,NaN
5,5/1/2026 12:00:00 AM,AA,AA,N102UW,1087,10561,BFL,"Bakersfield, CA",CA,11298,...,195.0,184.0,163.0,1271.0,6,34.0,0.0,0.0,0.0,0.0
6,5/1/2026 12:00:00 AM,AA,AA,N102UW,2065,11298,DFW,"Dallas/Fort Worth, TX",TX,11066,...,149.0,125.0,105.0,926.0,4,6.0,0.0,0.0,0.0,39.0
7,5/1/2026 12:00:00 AM,AA,AA,N102UW,2434,11298,DFW,"Dallas/Fort Worth, TX",TX,13577,...,159.0,136.0,118.0,1048.0,5,NaN,NaN,NaN,NaN,NaN
8,5/1/2026 12:00:00 AM,AA,AA,N102UW,2434,13577,MYR,"Myrtle Beach, SC",SC,11298,...,192.0,197.0,176.0,1048.0,5,53.0,0.0,20.0,0.0,3.0
9,5/1/2026 12:00:00 AM,AA,AA,N103NN,2455,10721,BOS,"Boston, MA",MA,12892,...,387.0,361.0,330.0,2611.0,11,33.0,0.0,0.0,0.0,0.0


## 11. Inspección inicial de los tipos de datos

Pandas determina automáticamente el tipo de cada variable durante la lectura del CSV.

Esta inferencia inicial no se considerará definitiva. Los archivos CSV no almacenan información explícita sobre tipos, por lo que Pandas puede asignar tipos de mayor tamaño del necesario o interpretar variables de forma diferente a su significado real.

Por ejemplo:

- Una fecha puede ser interpretada inicialmente como texto.
- Un indicador binario puede ser almacenado como `float64` debido a la existencia de valores nulos.
- Los códigos de aeropuertos y aerolíneas pueden ser interpretados como objetos de texto.
- Variables numéricas pueden utilizar 64 bits aunque su rango permita utilizar tipos más pequeños.

La optimización de estos tipos se realizará posteriormente durante la normalización de los datos.

In [12]:
sample_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 40 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   FL_DATE              10 non-null     object 
 1   MKT_UNIQUE_CARRIER   10 non-null     object 
 2   OP_UNIQUE_CARRIER    10 non-null     object 
 3   TAIL_NUM             10 non-null     object 
 4   OP_CARRIER_FL_NUM    10 non-null     int64  
 5   ORIGIN_AIRPORT_ID    10 non-null     int64  
 6   ORIGIN               10 non-null     object 
 7   ORIGIN_CITY_NAME     10 non-null     object 
 8   ORIGIN_STATE_ABR     10 non-null     object 
 9   DEST_AIRPORT_ID      10 non-null     int64  
 10  DEST                 10 non-null     object 
 11  DEST_CITY_NAME       10 non-null     object 
 12  DEST_STATE_ABR       10 non-null     object 
 13  CRS_DEP_TIME         10 non-null     int64  
 14  DEP_TIME             10 non-null     int64  
 15  DEP_DELAY            10 non-null     float6

In [13]:
sample_df.dtypes

FL_DATE                 object
MKT_UNIQUE_CARRIER      object
OP_UNIQUE_CARRIER       object
TAIL_NUM                object
OP_CARRIER_FL_NUM        int64
ORIGIN_AIRPORT_ID        int64
ORIGIN                  object
ORIGIN_CITY_NAME        object
ORIGIN_STATE_ABR        object
DEST_AIRPORT_ID          int64
DEST                    object
DEST_CITY_NAME          object
DEST_STATE_ABR          object
CRS_DEP_TIME             int64
DEP_TIME                 int64
DEP_DELAY              float64
DEP_DELAY_NEW          float64
DEP_DEL15              float64
DEP_TIME_BLK            object
TAXI_OUT               float64
TAXI_IN                float64
CRS_ARR_TIME             int64
ARR_TIME                 int64
ARR_DELAY              float64
ARR_DELAY_NEW          float64
ARR_DEL15              float64
ARR_TIME_BLK            object
CANCELLED              float64
CANCELLATION_CODE      float64
DIVERTED               float64
CRS_ELAPSED_TIME       float64
ACTUAL_ELAPSED_TIME    float64
AIR_TIME

## 12. Evaluación inicial del consumo de memoria

Debido al volumen total de información, uno de los objetivos del proceso de preparación será optimizar el uso de memoria.

Se analiza el consumo de la muestra utilizando la opción `deep=True`, que incluye una estimación más precisa del espacio utilizado por las variables de texto.

Aunque esta cifra corresponde únicamente a una pequeña muestra, permite comenzar a identificar variables cuyo tipo de datos podrá optimizarse posteriormente.

In [14]:
memory_usage = sample_df.memory_usage(
    deep=True
)

print("Memoria utilizada por variable:")
display(memory_usage.to_frame("bytes"))

print(
    "\nMemoria total de la muestra: "
    f"{memory_usage.sum() / 1024:.2f} KB"
)

Memoria utilizada por variable:


,bytes
Index,132
FL_DATE,770
MKT_UNIQUE_CARRIER,590
OP_UNIQUE_CARRIER,590
TAIL_NUM,630
OP_CARRIER_FL_NUM,80
ORIGIN_AIRPORT_ID,80
ORIGIN,600
ORIGIN_CITY_NAME,724
ORIGIN_STATE_ABR,590



Memoria total de la muestra: 9.86 KB


## 13. Estimación del consumo de memoria del conjunto completo

Debido al volumen total de los datos, no es recomendable cargar simultáneamente todos los archivos CSV en memoria únicamente para calcular su consumo.

En su lugar, se realiza una lectura secuencial por bloques (`chunks`). Para cada bloque se calcula la memoria ocupada por el DataFrame mediante `memory_usage(deep=True)` y posteriormente se acumulan los resultados.

Este procedimiento permite estimar el consumo de memoria que requeriría el conjunto completo si todos los registros fueran cargados simultáneamente en un único DataFrame de Pandas, sin comprometer la memoria disponible del equipo.

La estimación se realiza utilizando los tipos de datos inferidos automáticamente por Pandas, por lo que posteriormente podrá compararse con el consumo obtenido después de optimizar los tipos de datos.

In [15]:
total_memory_bytes = 0
total_rows = 0
total_chunks = 0

for file_number, csv_file in enumerate(csv_files, start=1):

    print(
        f"Procesando archivo {file_number}/{len(csv_files)}: "
        f"{csv_file.name}"
    )

    reader = pd.read_csv(
        csv_file,
        chunksize=CHUNK_SIZE,
        low_memory=False
    )

    for chunk in reader:

        chunk_memory = chunk.memory_usage(
            deep=True
        ).sum()

        total_memory_bytes += chunk_memory
        total_rows += len(chunk)
        total_chunks += 1

Procesando archivo 1/53: T_ONTIME_MARKETING.csv
Procesando archivo 2/53: T_ONTIME_MARKETING.csv
Procesando archivo 3/53: T_ONTIME_MARKETING.csv
Procesando archivo 4/53: T_ONTIME_MARKETING.csv
Procesando archivo 5/53: T_ONTIME_MARKETING.csv
Procesando archivo 6/53: T_ONTIME_MARKETING.csv
Procesando archivo 7/53: T_ONTIME_MARKETING.csv
Procesando archivo 8/53: T_ONTIME_MARKETING.csv
Procesando archivo 9/53: T_ONTIME_MARKETING.csv
Procesando archivo 10/53: T_ONTIME_MARKETING.csv
Procesando archivo 11/53: T_ONTIME_MARKETING.csv
Procesando archivo 12/53: T_ONTIME_MARKETING.csv
Procesando archivo 13/53: T_ONTIME_MARKETING.csv
Procesando archivo 14/53: T_ONTIME_MARKETING.csv
Procesando archivo 15/53: T_ONTIME_MARKETING.csv
Procesando archivo 16/53: T_ONTIME_MARKETING.csv
Procesando archivo 17/53: T_ONTIME_MARKETING.csv
Procesando archivo 18/53: T_ONTIME_MARKETING.csv
Procesando archivo 19/53: T_ONTIME_MARKETING.csv
Procesando archivo 20/53: T_ONTIME_MARKETING.csv
Procesando archivo 21/53: T_O

### Estimación conjunta del consumo de memoria

Para evitar realizar dos lecturas completas del conjunto de datos, se calcula simultáneamente:

- Número total de registros.
- Consumo total estimado de memoria.
- Consumo medio por registro.
- Consumo de memoria por variable.

El procesamiento continúa realizándose mediante bloques de 100.000 registros para mantener controlado el consumo de recursos.

In [16]:
total_memory_bytes = 0
total_rows = 0
total_chunks = 0

memory_by_column = {}

for file_number, csv_file in enumerate(
    csv_files,
    start=1
):

    print(
        f"[{file_number:02d}/{len(csv_files)}] "
        f"{csv_file.name}"
    )

    reader = pd.read_csv(
        csv_file,
        chunksize=CHUNK_SIZE,
        low_memory=False
    )

    for chunk in reader:

        memory = chunk.memory_usage(
            deep=True,
            index=False
        )

        # Memoria total
        total_memory_bytes += memory.sum()

        # Filas
        total_rows += len(chunk)

        # Número de bloques
        total_chunks += 1

        # Memoria por columna
        for column, bytes_used in memory.items():

            memory_by_column[column] = (
                memory_by_column.get(column, 0)
                + bytes_used
            )

[01/53] T_ONTIME_MARKETING.csv
[02/53] T_ONTIME_MARKETING.csv
[03/53] T_ONTIME_MARKETING.csv
[04/53] T_ONTIME_MARKETING.csv
[05/53] T_ONTIME_MARKETING.csv
[06/53] T_ONTIME_MARKETING.csv
[07/53] T_ONTIME_MARKETING.csv
[08/53] T_ONTIME_MARKETING.csv
[09/53] T_ONTIME_MARKETING.csv
[10/53] T_ONTIME_MARKETING.csv
[11/53] T_ONTIME_MARKETING.csv
[12/53] T_ONTIME_MARKETING.csv
[13/53] T_ONTIME_MARKETING.csv
[14/53] T_ONTIME_MARKETING.csv
[15/53] T_ONTIME_MARKETING.csv
[16/53] T_ONTIME_MARKETING.csv
[17/53] T_ONTIME_MARKETING.csv
[18/53] T_ONTIME_MARKETING.csv
[19/53] T_ONTIME_MARKETING.csv
[20/53] T_ONTIME_MARKETING.csv
[21/53] T_ONTIME_MARKETING.csv
[22/53] T_ONTIME_MARKETING.csv
[23/53] T_ONTIME_MARKETING.csv
[24/53] T_ONTIME_MARKETING.csv
[25/53] T_ONTIME_MARKETING.csv
[26/53] T_ONTIME_MARKETING.csv
[27/53] T_ONTIME_MARKETING.csv
[28/53] T_ONTIME_MARKETING.csv
[29/53] T_ONTIME_MARKETING.csv
[30/53] T_ONTIME_MARKETING.csv
[31/53] T_ONTIME_MARKETING.csv
[32/53] T_ONTIME_MARKETING.csv
[33/53] 

In [17]:
total_memory_gb = (
    total_memory_bytes
    / (1024 ** 3)
)

memory_per_row = (
    total_memory_bytes
    / total_rows
)

print("\nRESUMEN")
print("-" * 40)

print(
    f"Archivos       : "
    f"{len(csv_files):,}"
)

print(
    f"Chunks         : "
    f"{total_chunks:,}"
)

print(
    f"Filas          : "
    f"{total_rows:,}"
)

print(
    f"Memoria total  : "
    f"{total_memory_gb:,.2f} GB"
)

print(
    f"Bytes por fila : "
    f"{memory_per_row:,.2f}"
)



RESUMEN
----------------------------------------
Archivos       : 53
Chunks         : 354
Filas          : 32,761,129
Memoria total  : 31.04 GB
Bytes por fila : 1,017.42


In [18]:
memory_columns_df = pd.DataFrame(
    {
        "variable": memory_by_column.keys(),
        "memory_bytes": memory_by_column.values()
    }
)

memory_columns_df["memory_mb"] = (
    memory_columns_df["memory_bytes"]
    / (1024 ** 2)
)

memory_columns_df["percentage"] = (
    memory_columns_df["memory_bytes"]
    / total_memory_bytes
    * 100
)

memory_columns_df = (
    memory_columns_df
    .sort_values(
        "memory_bytes",
        ascending=False
    )
    .reset_index(drop=True)
)

memory_columns_df

,variable,memory_bytes,memory_mb,percentage
0,FL_DATE,2553173690,2434.896173,7.659857
1,DEST_CITY_NAME,2295931726,2189.571119,6.888097
2,ORIGIN_CITY_NAME,2295930506,2189.569956,6.888093
3,DEP_TIME_BLK,2162234514,2062.067522,6.486988
4,ARR_TIME_BLK,2162234514,2062.067522,6.486988
5,TAIL_NUM,2059690333,1964.273770,6.179342
6,DEST,1965667740,1874.606838,5.897262
7,ORIGIN,1965667740,1874.606838,5.897262
8,DEST_STATE_ABR,1932906611,1843.363391,5.798974
9,ORIGIN_STATE_ABR,1932906611,1843.363391,5.798974


## 14. Perfilado global de calidad de los datos

Una vez validada la estructura de los archivos y estimado su consumo de memoria, se realiza un análisis global de calidad.

El objetivo es identificar problemas potenciales antes de transformar los datos. Para ello se analizarán:

- Número total de registros.
- Valores nulos por variable.
- Porcentaje de valores nulos.
- Número de valores únicos.
- Tipos de datos inferidos.
- Valores mínimos y máximos para variables numéricas.
- Posibles inconsistencias entre variables relacionadas.

Debido al tamaño del conjunto de datos, el análisis se realizará nuevamente mediante procesamiento por bloques.

In [19]:
total_rows = 0
null_counts = {}
unique_values = {}
numeric_min = {}
numeric_max = {}

for file_number, csv_file in enumerate(csv_files, start=1):

    print(
        f"[{file_number:02d}/{len(csv_files)}] "
        f"{csv_file.name}"
    )

    reader = pd.read_csv(
        csv_file,
        chunksize=CHUNK_SIZE,
        low_memory=False
    )

    for chunk in reader:

        total_rows += len(chunk)

        # -------------------------------
        # Valores nulos
        # -------------------------------
        chunk_nulls = chunk.isna().sum()

        for column, count in chunk_nulls.items():
            null_counts[column] = (
                null_counts.get(column, 0)
                + int(count)
            )

        # -------------------------------
        # Valores únicos
        # -------------------------------
        for column in chunk.columns:

            values = chunk[column].dropna().unique()

            if column not in unique_values:
                unique_values[column] = set()

            unique_values[column].update(values)

        # -------------------------------
        # Mínimos y máximos numéricos
        # -------------------------------
        numeric_columns = chunk.select_dtypes(
            include="number"
        ).columns

        for column in numeric_columns:

            current_min = chunk[column].min()
            current_max = chunk[column].max()

            if pd.notna(current_min):
                numeric_min[column] = min(
                    numeric_min.get(
                        column,
                        current_min
                    ),
                    current_min
                )

            if pd.notna(current_max):
                numeric_max[column] = max(
                    numeric_max.get(
                        column,
                        current_max
                    ),
                    current_max
                )

[01/53] T_ONTIME_MARKETING.csv
[02/53] T_ONTIME_MARKETING.csv
[03/53] T_ONTIME_MARKETING.csv
[04/53] T_ONTIME_MARKETING.csv
[05/53] T_ONTIME_MARKETING.csv
[06/53] T_ONTIME_MARKETING.csv
[07/53] T_ONTIME_MARKETING.csv
[08/53] T_ONTIME_MARKETING.csv
[09/53] T_ONTIME_MARKETING.csv
[10/53] T_ONTIME_MARKETING.csv
[11/53] T_ONTIME_MARKETING.csv
[12/53] T_ONTIME_MARKETING.csv
[13/53] T_ONTIME_MARKETING.csv
[14/53] T_ONTIME_MARKETING.csv
[15/53] T_ONTIME_MARKETING.csv
[16/53] T_ONTIME_MARKETING.csv
[17/53] T_ONTIME_MARKETING.csv
[18/53] T_ONTIME_MARKETING.csv
[19/53] T_ONTIME_MARKETING.csv
[20/53] T_ONTIME_MARKETING.csv
[21/53] T_ONTIME_MARKETING.csv
[22/53] T_ONTIME_MARKETING.csv
[23/53] T_ONTIME_MARKETING.csv
[24/53] T_ONTIME_MARKETING.csv
[25/53] T_ONTIME_MARKETING.csv
[26/53] T_ONTIME_MARKETING.csv
[27/53] T_ONTIME_MARKETING.csv
[28/53] T_ONTIME_MARKETING.csv
[29/53] T_ONTIME_MARKETING.csv
[30/53] T_ONTIME_MARKETING.csv
[31/53] T_ONTIME_MARKETING.csv
[32/53] T_ONTIME_MARKETING.csv
[33/53] 

### 14.1 Análisis de valores faltantes

Los valores faltantes no deben interpretarse automáticamente como errores.

En este dataset existen variables cuyo valor solo está disponible bajo determinadas circunstancias. Por ejemplo, las causas de retraso únicamente tienen información en determinados vuelos con retrasos, mientras que el código de cancelación solo tiene sentido para vuelos cancelados.

Por esta razón, primero se cuantifica la presencia de valores nulos antes de decidir cómo tratarlos.

In [20]:
null_profile = pd.DataFrame(
    {
        "variable": null_counts.keys(),
        "null_count": null_counts.values()
    }
)

null_profile["null_percentage"] = (
    null_profile["null_count"]
    / total_rows
    * 100
)

null_profile = (
    null_profile
    .sort_values(
        "null_percentage",
        ascending=False
    )
    .reset_index(drop=True)
)

null_profile

,variable,null_count,null_percentage
0,CANCELLATION_CODE,32181577,98.230977
1,LATE_AIRCRAFT_DELAY,25997025,79.353263
2,SECURITY_DELAY,25997025,79.353263
3,NAS_DELAY,25997025,79.353263
4,WEATHER_DELAY,25997025,79.353263
5,CARRIER_DELAY,25997025,79.353263
6,AIR_TIME,662097,2.020983
7,ACTUAL_ELAPSED_TIME,662097,2.020983
8,ARR_DELAY,662088,2.020956
9,ARR_DEL15,662088,2.020956


### 14.2 Cardinalidad de las variables

La cardinalidad representa el número de valores distintos presentes en una variable.

Este análisis resulta especialmente útil para identificar:

- Variables binarias.
- Variables categóricas con pocas categorías.
- Variables categóricas de alta cardinalidad.
- Posibles identificadores.
- Variables candidatas a codificación posterior.

Una cardinalidad elevada puede tener implicaciones importantes en modelos de Machine Learning, especialmente cuando las variables categóricas deben transformarse.

In [21]:
cardinality_profile = pd.DataFrame(
    {
        "variable": unique_values.keys(),
        "unique_values": [
            len(values)
            for values in unique_values.values()
        ]
    }
)

cardinality_profile = (
    cardinality_profile
    .sort_values(
        "unique_values",
        ascending=False
    )
    .reset_index(drop=True)
)

cardinality_profile

,variable,unique_values
0,TAIL_NUM,7492
1,OP_CARRIER_FL_NUM,7083
2,ARR_DELAY,2404
3,DEP_DELAY,2384
4,DEP_DELAY_NEW,2300
5,ARR_DELAY_NEW,2298
6,CARRIER_DELAY,2178
7,DISTANCE,1814
8,LATE_AIRCRAFT_DELAY,1710
9,FL_DATE,1612


### 14.3 Análisis de rangos numéricos

Se analizan los valores mínimos y máximos de las variables numéricas.

El objetivo es identificar valores potencialmente anómalos y comprobar que las variables se encuentran dentro de rangos lógicamente posibles.

Este análisis no implica eliminar automáticamente valores extremos. Un retraso muy elevado, por ejemplo, puede representar un evento real y no necesariamente un error.

La detección y tratamiento de valores atípicos se realizará posteriormente teniendo en cuenta tanto criterios estadísticos como el significado operativo de las variables.

In [22]:
range_profile = pd.DataFrame(
    {
        "variable": numeric_min.keys(),
        "minimum": [
            numeric_min[column]
            for column in numeric_min
        ],
        "maximum": [
            numeric_max.get(column)
            for column in numeric_min
        ]
    }
)

range_profile

,variable,minimum,maximum
0,OP_CARRIER_FL_NUM,1.0,9914.0
1,ORIGIN_AIRPORT_ID,10135.0,16869.0
2,DEST_AIRPORT_ID,10135.0,16869.0
3,CRS_DEP_TIME,1.0,2400.0
4,DEP_TIME,1.0,2400.0
5,DEP_DELAY,-115.0,7223.0
6,DEP_DELAY_NEW,0.0,7223.0
7,DEP_DEL15,0.0,1.0
8,TAXI_OUT,1.0,1274.0
9,TAXI_IN,1.0,1318.0


## 15. Generación de variables temporales a partir de `FL_DATE`

El conjunto de datos contiene una única variable temporal original: `FL_DATE`.

Para evitar redundancia, las variables de año, trimestre, mes, día del mes y día de la semana no se descargaron directamente desde la fuente. En su lugar, se generan a partir de la fecha del vuelo durante la fase de preparación de datos.

Este procedimiento forma parte de la ingeniería de características y permite mantener una única fuente temporal original, evitando almacenar información redundante.

Las variables generadas son:

- `YEAR`: año del vuelo.
- `QUARTER`: trimestre del año.
- `MONTH`: mes.
- `DAY_OF_MONTH`: día del mes.
- `DAY_OF_WEEK`: día de la semana.

Se utiliza la convención ISO para el día de la semana:

- 1 = lunes
- 2 = martes
- ...
- 7 = domingo

### Conversión explícita de la variable `FL_DATE`

La variable `FL_DATE` se encuentra almacenada como texto con un formato de fecha y hora similar a:

`1/1/2026 12:00:00 AM`

Para evitar que Pandas tenga que inferir automáticamente el formato de cada registro, se especifica explícitamente el patrón utilizado por la fuente.

Esto mejora:

- La velocidad de procesamiento.
- La consistencia en la interpretación de las fechas.
- La reproducibilidad del proceso de transformación.

La hora incluida en el campo no aporta información adicional para este proyecto, ya que corresponde a medianoche. Posteriormente se conservará la fecha como variable temporal principal.

In [23]:
chunk["FL_DATE"] = pd.to_datetime(
    chunk["FL_DATE"],
    format="%m/%d/%Y %I:%M:%S %p",
    errors="coerce"
)

In [24]:
chunk["FL_DATE"] = pd.to_datetime(
    chunk["FL_DATE"],
    errors="coerce"
)

chunk["YEAR"] = chunk["FL_DATE"].dt.year
chunk["QUARTER"] = chunk["FL_DATE"].dt.quarter
chunk["MONTH"] = chunk["FL_DATE"].dt.month
chunk["DAY_OF_MONTH"] = chunk["FL_DATE"].dt.day
chunk["DAY_OF_WEEK"] = chunk["FL_DATE"].dt.dayofweek + 1

In [25]:
for chunk in reader:

    # Normalización de encabezados
    chunk.columns = (
        chunk.columns
        .str.strip()
        .str.upper()
    )

    # Conversión de la fecha
    chunk["FL_DATE"] = pd.to_datetime(
        chunk["FL_DATE"],
        errors="coerce"
    )

    # Feature engineering temporal
    chunk["YEAR"] = chunk["FL_DATE"].dt.year
    chunk["QUARTER"] = chunk["FL_DATE"].dt.quarter
    chunk["MONTH"] = chunk["FL_DATE"].dt.month
    chunk["DAY_OF_MONTH"] = chunk["FL_DATE"].dt.day
    chunk["DAY_OF_WEEK"] = (
        chunk["FL_DATE"].dt.dayofweek + 1
    )

    # A partir de aquí ya podemos validar
    validation_errors["invalid_month"] += (
        ~chunk["MONTH"]
        .dropna()
        .between(1, 12)
    ).sum()

    validation_errors["invalid_quarter"] += (
        ~chunk["QUARTER"]
        .dropna()
        .between(1, 4)
    ).sum()

    validation_errors["invalid_day_of_week"] += (
        ~chunk["DAY_OF_WEEK"]
        .dropna()
        .between(1, 7)
    ).sum()

In [26]:
chunk

,FL_DATE,MKT_UNIQUE_CARRIER,OP_UNIQUE_CARRIER,TAIL_NUM,OP_CARRIER_FL_NUM,ORIGIN_AIRPORT_ID,ORIGIN,ORIGIN_CITY_NAME,ORIGIN_STATE_ABR,DEST_AIRPORT_ID,...,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY,YEAR,QUARTER,MONTH,DAY_OF_MONTH,DAY_OF_WEEK
500000,2022-12-27,UA,UA,N27267,512,14683,SAT,"San Antonio, TX",TX,11618,...,NaN,NaN,NaN,NaN,NaN,2022,4,12,27,2
500001,2022-12-27,UA,UA,N27268,1338,13930,ORD,"Chicago, IL",IL,14122,...,NaN,NaN,NaN,NaN,NaN,2022,4,12,27,2
500002,2022-12-27,UA,UA,N27268,2438,13930,ORD,"Chicago, IL",IL,11697,...,0.0,0.0,28.0,0.0,0.0,2022,4,12,27,2
500003,2022-12-27,UA,UA,N27268,303,11697,FLL,"Fort Lauderdale, FL",FL,11618,...,0.0,0.0,20.0,0.0,21.0,2022,4,12,27,2
500004,2022-12-27,UA,UA,N27268,623,14122,PIT,"Pittsburgh, PA",PA,13930,...,NaN,NaN,NaN,NaN,NaN,2022,4,12,27,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
578316,2022-12-31,WN,WN,N969WN,3180,10821,BWI,"Baltimore, MD",MD,11066,...,NaN,NaN,NaN,NaN,NaN,2022,4,12,31,6
578317,2022-12-31,WN,WN,N969WN,3385,10994,CHS,"Charleston, SC",SC,10821,...,NaN,NaN,NaN,NaN,NaN,2022,4,12,31,6
578318,2022-12-31,WN,WN,N969WN,3385,11292,DEN,"Denver, CO",CO,10994,...,NaN,NaN,NaN,NaN,NaN,2022,4,12,31,6
578319,2022-12-31,WN,WN,N969WN,3385,14679,SAN,"San Diego, CA",CA,11292,...,NaN,NaN,NaN,NaN,NaN,2022,4,12,31,6


## 16. Ajuste y normalización de variables

La lógica de normalización quedó consolidada en `src/preprocessing.py`.

Este notebook conserva la explicación académica y las pruebas del proceso, mientras que
la implementación reutilizable se mantiene en un único módulo Python. Esto permite usar
las mismas reglas en futuras cargas históricas y evita mantener versiones distintas de
la función en varios notebooks.


### 16.1 Clasificación de las variables

Las variables se clasifican según su significado y no únicamente por el tipo técnico con
el que aparecen en el CSV. Las listas se importan desde `src.preprocessing`.

Esta clasificación determina qué conversión estructural se aplica a cada grupo y será
reutilizada en todo archivo nuevo que ingrese al histórico.


### 16.2 Esquema analítico final

`FINAL_COLUMN_ORDER`, definido en `src.preprocessing`, establece el contrato de salida
del preprocesamiento: 40 variables originales más 5 variables temporales derivadas.

Centralizar este esquema evita que diferentes notebooks utilicen órdenes o conjuntos de
columnas distintos.


### 16.3 Prueba de `normalize_chunk()`

Antes de aplicar el pipeline de forma masiva se prueba la función reutilizable sobre una
muestra. El objetivo es confirmar que el contrato del módulo se cumple sin volver a
definir la lógica en el notebook.


In [27]:
# Se utiliza una muestra pequeña para validar la función sin procesar
# nuevamente todo el histórico.
test_file = csv_files[0]

test_df = pd.read_csv(
    test_file,
    nrows=1000,
    low_memory=False,
)

# normalize_chunk() aplica tipos, fecha, variables temporales y orden final.
test_df = normalize_chunk(test_df)

print(f"Archivo utilizado: {test_file}")
print(f"Dimensiones del dataset normalizado: {test_df.shape}")


Archivo utilizado: G:\My Drive\MASTER Big Data\TFM\data\raw\Flights\T_ONTIME_MARKETING_20260807_125204\T_ONTIME_MARKETING.csv
Dimensiones del dataset normalizado: (1000, 45)


### 16.4 Validación del resultado

Una vez aplicada la función de normalización, se verifica que la estructura resultante sea coherente.

Se comprueba:

- Número total de columnas.
- Tipos de datos asignados.
- Correcta generación de las variables temporales.
- Orden final de las variables.

Esta validación permite confirmar que la función de normalización está preparada para utilizarse posteriormente sobre todos los archivos del proyecto.

In [28]:
print("Número de columnas:")
print(len(test_df.columns))

print("\nTipos de datos:")
schema_df = pd.DataFrame({
    "variable": test_df.columns,
    "dtype": test_df.dtypes.astype(str).values
})

display(schema_df)

print("\nVariables temporales:")
display(
    test_df[
        [
            "FL_DATE",
            "YEAR",
            "QUARTER",
            "MONTH",
            "DAY_OF_MONTH",
            "DAY_OF_WEEK"
        ]
    ].head(10)
)

print("\nOrden de las columnas:")
for i, column in enumerate(test_df.columns, start=1):
    print(f"{i:02d}. {column}")

Número de columnas:
45

Tipos de datos:


,variable,dtype
0,FL_DATE,datetime64[ns]
1,YEAR,Int16
2,QUARTER,Int8
3,MONTH,Int8
4,DAY_OF_MONTH,Int8
5,DAY_OF_WEEK,Int8
6,MKT_UNIQUE_CARRIER,string
7,OP_UNIQUE_CARRIER,string
8,TAIL_NUM,string
9,OP_CARRIER_FL_NUM,Int32



Variables temporales:


,FL_DATE,YEAR,QUARTER,MONTH,DAY_OF_MONTH,DAY_OF_WEEK
0,2026-05-01,2026,2,5,1,5
1,2026-05-01,2026,2,5,1,5
2,2026-05-01,2026,2,5,1,5
3,2026-05-01,2026,2,5,1,5
4,2026-05-01,2026,2,5,1,5
5,2026-05-01,2026,2,5,1,5
6,2026-05-01,2026,2,5,1,5
7,2026-05-01,2026,2,5,1,5
8,2026-05-01,2026,2,5,1,5
9,2026-05-01,2026,2,5,1,5



Orden de las columnas:
01. FL_DATE
02. YEAR
03. QUARTER
04. MONTH
05. DAY_OF_MONTH
06. DAY_OF_WEEK
07. MKT_UNIQUE_CARRIER
08. OP_UNIQUE_CARRIER
09. TAIL_NUM
10. OP_CARRIER_FL_NUM
11. ORIGIN_AIRPORT_ID
12. ORIGIN
13. ORIGIN_CITY_NAME
14. ORIGIN_STATE_ABR
15. DEST_AIRPORT_ID
16. DEST
17. DEST_CITY_NAME
18. DEST_STATE_ABR
19. CRS_DEP_TIME
20. DEP_TIME
21. DEP_DELAY
22. DEP_DELAY_NEW
23. DEP_DEL15
24. DEP_TIME_BLK
25. TAXI_OUT
26. TAXI_IN
27. CRS_ARR_TIME
28. ARR_TIME
29. ARR_DELAY
30. ARR_DELAY_NEW
31. ARR_DEL15
32. ARR_TIME_BLK
33. CANCELLED
34. CANCELLATION_CODE
35. DIVERTED
36. CRS_ELAPSED_TIME
37. ACTUAL_ELAPSED_TIME
38. AIR_TIME
39. DISTANCE
40. DISTANCE_GROUP
41. CARRIER_DELAY
42. WEATHER_DELAY
43. NAS_DELAY
44. SECURITY_DELAY
45. LATE_AIRCRAFT_DELAY


## 17. Análisis de calidad de los datos

Una vez definida y validada la normalización estructural, se procede a evaluar la calidad del conjunto de datos.

Esta fase tiene como objetivo identificar problemas potenciales antes de realizar cualquier limpieza o imputación.

El primer análisis se centra en los valores faltantes.

Debido al volumen total del dataset, el cálculo se realiza recorriendo los archivos por bloques (`chunks`) y aplicando previamente la función `normalize_chunk()` definida en el apartado 16.3.

El objetivo es obtener, para cada variable:

- Número total de valores faltantes.
- Porcentaje de valores faltantes respecto al total de registros.

En esta etapa no se modifica ningún dato. Los resultados servirán posteriormente para decidir qué valores faltantes representan un problema real y cuáles son valores estructurales derivados de la lógica operacional de los vuelos.

In [29]:
total_rows = 0

null_counts = {
    column: 0
    for column in FINAL_COLUMN_ORDER
}

for file_number, csv_file in enumerate(
    csv_files,
    start=1
):

    print(
        f"[{file_number:02d}/{len(csv_files)}] "
        f"{csv_file.name}"
    )

    reader = pd.read_csv(
        csv_file,
        chunksize=CHUNK_SIZE,
        low_memory=False
    )

    for chunk in reader:

        # Aplicar la normalización estructural
        chunk = normalize_chunk(chunk)

        # Acumular número total de registros
        total_rows += len(chunk)

        # Contabilizar nulos por variable
        chunk_nulls = chunk.isna().sum()

        for column, count in chunk_nulls.items():

            null_counts[column] += int(count)


# ------------------------------------------------------------
# Construcción del resumen
# ------------------------------------------------------------

null_profile_df = pd.DataFrame({
    "variable": null_counts.keys(),
    "null_count": null_counts.values()
})

null_profile_df["null_percentage"] = (
    null_profile_df["null_count"]
    / total_rows
    * 100
)

null_profile_df = (
    null_profile_df
    .sort_values(
        "null_percentage",
        ascending=False
    )
    .reset_index(drop=True)
)


print("\nResumen del análisis:")
print(f"Total de registros analizados: {total_rows:,}")

display(null_profile_df)

[01/53] T_ONTIME_MARKETING.csv
[02/53] T_ONTIME_MARKETING.csv
[03/53] T_ONTIME_MARKETING.csv
[04/53] T_ONTIME_MARKETING.csv
[05/53] T_ONTIME_MARKETING.csv
[06/53] T_ONTIME_MARKETING.csv
[07/53] T_ONTIME_MARKETING.csv
[08/53] T_ONTIME_MARKETING.csv
[09/53] T_ONTIME_MARKETING.csv
[10/53] T_ONTIME_MARKETING.csv
[11/53] T_ONTIME_MARKETING.csv
[12/53] T_ONTIME_MARKETING.csv
[13/53] T_ONTIME_MARKETING.csv
[14/53] T_ONTIME_MARKETING.csv
[15/53] T_ONTIME_MARKETING.csv
[16/53] T_ONTIME_MARKETING.csv
[17/53] T_ONTIME_MARKETING.csv
[18/53] T_ONTIME_MARKETING.csv
[19/53] T_ONTIME_MARKETING.csv
[20/53] T_ONTIME_MARKETING.csv
[21/53] T_ONTIME_MARKETING.csv
[22/53] T_ONTIME_MARKETING.csv
[23/53] T_ONTIME_MARKETING.csv
[24/53] T_ONTIME_MARKETING.csv
[25/53] T_ONTIME_MARKETING.csv
[26/53] T_ONTIME_MARKETING.csv
[27/53] T_ONTIME_MARKETING.csv
[28/53] T_ONTIME_MARKETING.csv
[29/53] T_ONTIME_MARKETING.csv
[30/53] T_ONTIME_MARKETING.csv
[31/53] T_ONTIME_MARKETING.csv
[32/53] T_ONTIME_MARKETING.csv
[33/53] 

,variable,null_count,null_percentage
0,CANCELLATION_CODE,32181577,98.230977
1,LATE_AIRCRAFT_DELAY,25997025,79.353263
2,SECURITY_DELAY,25997025,79.353263
3,NAS_DELAY,25997025,79.353263
4,WEATHER_DELAY,25997025,79.353263
5,CARRIER_DELAY,25997025,79.353263
6,AIR_TIME,662097,2.020983
7,ACTUAL_ELAPSED_TIME,662097,2.020983
8,ARR_DEL15,662088,2.020956
9,ARR_DELAY_NEW,662088,2.020956


### 17.2 Análisis del significado de los valores faltantes

La presencia de valores faltantes no implica necesariamente un problema de calidad.

En este conjunto de datos existen variables cuyo valor únicamente tiene sentido cuando ocurre un determinado evento.

Por ejemplo:

- `CANCELLATION_CODE` solo tiene significado cuando el vuelo ha sido cancelado.
- Las variables relacionadas con causas de retraso solo se informan en determinados vuelos con retrasos.
- Algunas variables operativas pueden no existir cuando el vuelo ha sido cancelado o desviado.

Por esta razón, se analiza la relación entre los valores faltantes y el estado operacional del vuelo antes de decidir cualquier tratamiento.

In [63]:
missing_context['cancelled_flights']

579552

In [30]:
missing_context = {
    "cancelled_flights": 0,
    "cancelled_without_code": 0,
    "not_cancelled_with_code": 0,

    "arr_delayed_15": 0,
    "arr_delayed_15_without_causes": 0,

    "not_arr_delayed_15": 0,
    "not_arr_delayed_15_with_causes": 0,

    "diverted_flights": 0,

    "cancelled_missing_dep_time": 0,
    "cancelled_missing_arr_time": 0,
}

CAUSE_COLUMNS = [
    "CARRIER_DELAY",
    "WEATHER_DELAY",
    "NAS_DELAY",
    "SECURITY_DELAY",
    "LATE_AIRCRAFT_DELAY"
]

for file_number, csv_file in enumerate(
    csv_files,
    start=1
):

    print(
        f"[{file_number:02d}/{len(csv_files)}] "
        f"{csv_file.name}"
    )

    reader = pd.read_csv(
        csv_file,
        chunksize=CHUNK_SIZE,
        low_memory=False
    )

    for chunk in reader:

        chunk = normalize_chunk(chunk)

        # ----------------------------------------------------
        # Cancelaciones
        # ----------------------------------------------------

        cancelled_mask = (
            chunk["CANCELLED"] == 1
        )

        not_cancelled_mask = (
            chunk["CANCELLED"] == 0
        )

        missing_context["cancelled_flights"] += (
            cancelled_mask.sum()
        )

        missing_context[
            "cancelled_without_code"
        ] += (
            cancelled_mask
            & chunk["CANCELLATION_CODE"].isna()
        ).sum()

        missing_context[
            "not_cancelled_with_code"
        ] += (
            not_cancelled_mask
            & chunk["CANCELLATION_CODE"].notna()
        ).sum()

        # ----------------------------------------------------
        # Retrasos de llegada >= 15 minutos
        # ----------------------------------------------------

        delayed_mask = (
            chunk["ARR_DEL15"] == 1
        )

        not_delayed_mask = (
            chunk["ARR_DEL15"] == 0
        )

        missing_context["arr_delayed_15"] += (
            delayed_mask.sum()
        )

        missing_context["not_arr_delayed_15"] += (
            not_delayed_mask.sum()
        )

        all_causes_missing = (
            chunk[CAUSE_COLUMNS]
            .isna()
            .all(axis=1)
        )

        any_cause_present = (
            chunk[CAUSE_COLUMNS]
            .notna()
            .any(axis=1)
        )

        missing_context[
            "arr_delayed_15_without_causes"
        ] += (
            delayed_mask
            & all_causes_missing
        ).sum()

        missing_context[
            "not_arr_delayed_15_with_causes"
        ] += (
            not_delayed_mask
            & any_cause_present
        ).sum()

        # ----------------------------------------------------
        # Desvíos
        # ----------------------------------------------------

        diverted_mask = (
            chunk["DIVERTED"] == 1
        )

        missing_context["diverted_flights"] += (
            diverted_mask.sum()
        )

        # ----------------------------------------------------
        # Horarios en vuelos cancelados
        # ----------------------------------------------------

        missing_context[
            "cancelled_missing_dep_time"
        ] += (
            cancelled_mask
            & chunk["DEP_TIME"].isna()
        ).sum()

        missing_context[
            "cancelled_missing_arr_time"
        ] += (
            cancelled_mask
            & chunk["ARR_TIME"].isna()
        ).sum()


missing_context_df = pd.DataFrame(
    missing_context.items(),
    columns=[
        "condition",
        "count"
    ]
)

display(missing_context_df)

[01/53] T_ONTIME_MARKETING.csv
[02/53] T_ONTIME_MARKETING.csv
[03/53] T_ONTIME_MARKETING.csv
[04/53] T_ONTIME_MARKETING.csv
[05/53] T_ONTIME_MARKETING.csv
[06/53] T_ONTIME_MARKETING.csv
[07/53] T_ONTIME_MARKETING.csv
[08/53] T_ONTIME_MARKETING.csv
[09/53] T_ONTIME_MARKETING.csv
[10/53] T_ONTIME_MARKETING.csv
[11/53] T_ONTIME_MARKETING.csv
[12/53] T_ONTIME_MARKETING.csv
[13/53] T_ONTIME_MARKETING.csv
[14/53] T_ONTIME_MARKETING.csv
[15/53] T_ONTIME_MARKETING.csv
[16/53] T_ONTIME_MARKETING.csv
[17/53] T_ONTIME_MARKETING.csv
[18/53] T_ONTIME_MARKETING.csv
[19/53] T_ONTIME_MARKETING.csv
[20/53] T_ONTIME_MARKETING.csv
[21/53] T_ONTIME_MARKETING.csv
[22/53] T_ONTIME_MARKETING.csv
[23/53] T_ONTIME_MARKETING.csv
[24/53] T_ONTIME_MARKETING.csv
[25/53] T_ONTIME_MARKETING.csv
[26/53] T_ONTIME_MARKETING.csv
[27/53] T_ONTIME_MARKETING.csv
[28/53] T_ONTIME_MARKETING.csv
[29/53] T_ONTIME_MARKETING.csv
[30/53] T_ONTIME_MARKETING.csv
[31/53] T_ONTIME_MARKETING.csv
[32/53] T_ONTIME_MARKETING.csv
[33/53] 

,condition,count
0,cancelled_flights,579552
1,cancelled_without_code,0
2,not_cancelled_with_code,0
3,arr_delayed_15,6764107
4,arr_delayed_15_without_causes,3
5,not_arr_delayed_15,25334934
6,not_arr_delayed_15_with_causes,0
7,diverted_flights,82533
8,cancelled_missing_dep_time,559793
9,cancelled_missing_arr_time,579552


### 17.3 Análisis de registros duplicados

Se analiza la posible existencia de registros duplicados.

En un conjunto de datos de vuelos no es suficiente utilizar únicamente el número de vuelo para identificar duplicados, ya que un mismo número puede repetirse en diferentes fechas o rutas.

Por este motivo, se define una combinación de variables que representa de forma razonable la identidad operacional de un vuelo:

- Fecha del vuelo.
- Aerolínea operadora.
- Número de vuelo.
- Aeropuerto de origen.
- Aeropuerto de destino.
- Hora programada de salida.

En esta etapa únicamente se contabilizan posibles duplicados. No se elimina ningún registro.

In [31]:
DUPLICATE_KEY = [
    "FL_DATE",
    "OP_UNIQUE_CARRIER",
    "OP_CARRIER_FL_NUM",
    "ORIGIN_AIRPORT_ID",
    "DEST_AIRPORT_ID",
    "CRS_DEP_TIME"
]

duplicate_count = 0
total_rows_duplicates = 0

seen_keys = set()

for file_number, csv_file in enumerate(
    csv_files,
    start=1
):

    print(
        f"[{file_number:02d}/{len(csv_files)}] "
        f"{csv_file.name}"
    )

    reader = pd.read_csv(
        csv_file,
        chunksize=CHUNK_SIZE,
        low_memory=False
    )

    for chunk in reader:

        chunk = normalize_chunk(chunk)

        total_rows_duplicates += len(chunk)

        keys = chunk[DUPLICATE_KEY].itertuples(
            index=False,
            name=None
        )

        for key in keys:

            if key in seen_keys:
                duplicate_count += 1
            else:
                seen_keys.add(key)


print(
    f"Registros analizados : "
    f"{total_rows_duplicates:,}"
)

print(
    f"Posibles duplicados  : "
    f"{duplicate_count:,}"
)

print(
    f"Porcentaje duplicado : "
    f"{duplicate_count / total_rows_duplicates * 100:.4f}%"
)

[01/53] T_ONTIME_MARKETING.csv
[02/53] T_ONTIME_MARKETING.csv
[03/53] T_ONTIME_MARKETING.csv
[04/53] T_ONTIME_MARKETING.csv
[05/53] T_ONTIME_MARKETING.csv
[06/53] T_ONTIME_MARKETING.csv
[07/53] T_ONTIME_MARKETING.csv
[08/53] T_ONTIME_MARKETING.csv
[09/53] T_ONTIME_MARKETING.csv
[10/53] T_ONTIME_MARKETING.csv
[11/53] T_ONTIME_MARKETING.csv
[12/53] T_ONTIME_MARKETING.csv
[13/53] T_ONTIME_MARKETING.csv
[14/53] T_ONTIME_MARKETING.csv
[15/53] T_ONTIME_MARKETING.csv
[16/53] T_ONTIME_MARKETING.csv
[17/53] T_ONTIME_MARKETING.csv
[18/53] T_ONTIME_MARKETING.csv
[19/53] T_ONTIME_MARKETING.csv
[20/53] T_ONTIME_MARKETING.csv
[21/53] T_ONTIME_MARKETING.csv
[22/53] T_ONTIME_MARKETING.csv
[23/53] T_ONTIME_MARKETING.csv
[24/53] T_ONTIME_MARKETING.csv
[25/53] T_ONTIME_MARKETING.csv
[26/53] T_ONTIME_MARKETING.csv
[27/53] T_ONTIME_MARKETING.csv
[28/53] T_ONTIME_MARKETING.csv
[29/53] T_ONTIME_MARKETING.csv
[30/53] T_ONTIME_MARKETING.csv
[31/53] T_ONTIME_MARKETING.csv
[32/53] T_ONTIME_MARKETING.csv
[33/53] 

### 17.4 Validación de rangos

Se comprueba que determinadas variables se encuentren dentro de rangos lógicamente válidos.

Estas validaciones permiten detectar errores estructurales o valores imposibles.

Entre otras comprobaciones se analizan:

- Mes entre 1 y 12.
- Trimestre entre 1 y 4.
- Día de la semana entre 1 y 7.
- Variables binarias limitadas a 0 o 1.
- Distancia no negativa.
- Tiempo de vuelo no negativo.
- Tiempos de taxi no negativos.

Los valores extremos que siguen siendo operacionalmente posibles no se consideran errores en esta etapa.

In [32]:
range_errors = {
    "invalid_month": 0,
    "invalid_quarter": 0,
    "invalid_day_of_week": 0,

    "invalid_dep_del15": 0,
    "invalid_arr_del15": 0,
    "invalid_cancelled": 0,
    "invalid_diverted": 0,

    "negative_distance": 0,
    "negative_air_time": 0,
    "negative_taxi_out": 0,
    "negative_taxi_in": 0
}

for file_number, csv_file in enumerate(
    csv_files,
    start=1
):

    print(
        f"[{file_number:02d}/{len(csv_files)}] "
        f"{csv_file.name}"
    )

    reader = pd.read_csv(
        csv_file,
        chunksize=CHUNK_SIZE,
        low_memory=False
    )

    for chunk in reader:

        chunk = normalize_chunk(chunk)

        range_errors["invalid_month"] += (
            ~chunk["MONTH"]
            .dropna()
            .between(1, 12)
        ).sum()

        range_errors["invalid_quarter"] += (
            ~chunk["QUARTER"]
            .dropna()
            .between(1, 4)
        ).sum()

        range_errors["invalid_day_of_week"] += (
            ~chunk["DAY_OF_WEEK"]
            .dropna()
            .between(1, 7)
        ).sum()

        for column, key in [
            ("DEP_DEL15", "invalid_dep_del15"),
            ("ARR_DEL15", "invalid_arr_del15"),
            ("CANCELLED", "invalid_cancelled"),
            ("DIVERTED", "invalid_diverted"),
        ]:

            range_errors[key] += (
                ~chunk[column]
                .dropna()
                .isin([0, 1])
            ).sum()

        range_errors["negative_distance"] += (
            chunk["DISTANCE"] < 0
        ).sum()

        range_errors["negative_air_time"] += (
            chunk["AIR_TIME"] < 0
        ).sum()

        range_errors["negative_taxi_out"] += (
            chunk["TAXI_OUT"] < 0
        ).sum()

        range_errors["negative_taxi_in"] += (
            chunk["TAXI_IN"] < 0
        ).sum()


range_errors_df = pd.DataFrame(
    range_errors.items(),
    columns=[
        "validation",
        "error_count"
    ]
)

display(range_errors_df)

[01/53] T_ONTIME_MARKETING.csv
[02/53] T_ONTIME_MARKETING.csv
[03/53] T_ONTIME_MARKETING.csv
[04/53] T_ONTIME_MARKETING.csv
[05/53] T_ONTIME_MARKETING.csv
[06/53] T_ONTIME_MARKETING.csv
[07/53] T_ONTIME_MARKETING.csv
[08/53] T_ONTIME_MARKETING.csv
[09/53] T_ONTIME_MARKETING.csv
[10/53] T_ONTIME_MARKETING.csv
[11/53] T_ONTIME_MARKETING.csv
[12/53] T_ONTIME_MARKETING.csv
[13/53] T_ONTIME_MARKETING.csv
[14/53] T_ONTIME_MARKETING.csv
[15/53] T_ONTIME_MARKETING.csv
[16/53] T_ONTIME_MARKETING.csv
[17/53] T_ONTIME_MARKETING.csv
[18/53] T_ONTIME_MARKETING.csv
[19/53] T_ONTIME_MARKETING.csv
[20/53] T_ONTIME_MARKETING.csv
[21/53] T_ONTIME_MARKETING.csv
[22/53] T_ONTIME_MARKETING.csv
[23/53] T_ONTIME_MARKETING.csv
[24/53] T_ONTIME_MARKETING.csv
[25/53] T_ONTIME_MARKETING.csv
[26/53] T_ONTIME_MARKETING.csv
[27/53] T_ONTIME_MARKETING.csv
[28/53] T_ONTIME_MARKETING.csv
[29/53] T_ONTIME_MARKETING.csv
[30/53] T_ONTIME_MARKETING.csv
[31/53] T_ONTIME_MARKETING.csv
[32/53] T_ONTIME_MARKETING.csv
[33/53] 

,validation,error_count
0,invalid_month,0
1,invalid_quarter,0
2,invalid_day_of_week,0
3,invalid_dep_del15,0
4,invalid_arr_del15,0
5,invalid_cancelled,0
6,invalid_diverted,0
7,negative_distance,0
8,negative_air_time,0
9,negative_taxi_out,0


### 17.5 Validación de consistencia lógica entre variables

Además de comprobar rangos individuales, se analizan relaciones lógicas entre variables.

Por ejemplo:

- Si `DEP_DELAY_NEW` es igual o superior a 15 minutos, `DEP_DEL15` debería indicar retraso.
- Si `ARR_DELAY_NEW` es igual o superior a 15 minutos, `ARR_DEL15` debería indicar retraso.
- Un vuelo marcado como cancelado no debería presentar determinados resultados operativos finales de forma ordinaria.

Estas comprobaciones permiten detectar registros que individualmente contienen valores válidos, pero cuya combinación resulta incoherente.

In [33]:
consistency_errors = {
    "dep_delay_indicator": 0,
    "arr_delay_indicator": 0,
    "cancelled_with_arrival_time": 0,
    "cancelled_with_air_time": 0
}

for file_number, csv_file in enumerate(
    csv_files,
    start=1
):

    print(
        f"[{file_number:02d}/{len(csv_files)}] "
        f"{csv_file.name}"
    )

    reader = pd.read_csv(
        csv_file,
        chunksize=CHUNK_SIZE,
        low_memory=False
    )

    for chunk in reader:

        chunk = normalize_chunk(chunk)

        # ----------------------------------------------------
        # DEP_DELAY_NEW vs DEP_DEL15
        # ----------------------------------------------------

        valid_dep = (
            chunk["DEP_DELAY_NEW"].notna()
            & chunk["DEP_DEL15"].notna()
        )

        expected_dep_del15 = (
            chunk.loc[
                valid_dep,
                "DEP_DELAY_NEW"
            ] >= 15
        )

        actual_dep_del15 = (
            chunk.loc[
                valid_dep,
                "DEP_DEL15"
            ] == 1
        )

        consistency_errors[
            "dep_delay_indicator"
        ] += (
            expected_dep_del15
            != actual_dep_del15
        ).sum()

        # ----------------------------------------------------
        # ARR_DELAY_NEW vs ARR_DEL15
        # ----------------------------------------------------

        valid_arr = (
            chunk["ARR_DELAY_NEW"].notna()
            & chunk["ARR_DEL15"].notna()
        )

        expected_arr_del15 = (
            chunk.loc[
                valid_arr,
                "ARR_DELAY_NEW"
            ] >= 15
        )

        actual_arr_del15 = (
            chunk.loc[
                valid_arr,
                "ARR_DEL15"
            ] == 1
        )

        consistency_errors[
            "arr_delay_indicator"
        ] += (
            expected_arr_del15
            != actual_arr_del15
        ).sum()

        # ----------------------------------------------------
        # Cancelaciones
        # ----------------------------------------------------

        cancelled = (
            chunk["CANCELLED"] == 1
        )

        consistency_errors[
            "cancelled_with_arrival_time"
        ] += (
            cancelled
            & chunk["ARR_TIME"].notna()
        ).sum()

        consistency_errors[
            "cancelled_with_air_time"
        ] += (
            cancelled
            & chunk["AIR_TIME"].notna()
        ).sum()


consistency_errors_df = pd.DataFrame(
    consistency_errors.items(),
    columns=[
        "validation",
        "error_count"
    ]
)

display(consistency_errors_df)

[01/53] T_ONTIME_MARKETING.csv
[02/53] T_ONTIME_MARKETING.csv
[03/53] T_ONTIME_MARKETING.csv
[04/53] T_ONTIME_MARKETING.csv
[05/53] T_ONTIME_MARKETING.csv
[06/53] T_ONTIME_MARKETING.csv
[07/53] T_ONTIME_MARKETING.csv
[08/53] T_ONTIME_MARKETING.csv
[09/53] T_ONTIME_MARKETING.csv
[10/53] T_ONTIME_MARKETING.csv
[11/53] T_ONTIME_MARKETING.csv
[12/53] T_ONTIME_MARKETING.csv
[13/53] T_ONTIME_MARKETING.csv
[14/53] T_ONTIME_MARKETING.csv
[15/53] T_ONTIME_MARKETING.csv
[16/53] T_ONTIME_MARKETING.csv
[17/53] T_ONTIME_MARKETING.csv
[18/53] T_ONTIME_MARKETING.csv
[19/53] T_ONTIME_MARKETING.csv
[20/53] T_ONTIME_MARKETING.csv
[21/53] T_ONTIME_MARKETING.csv
[22/53] T_ONTIME_MARKETING.csv
[23/53] T_ONTIME_MARKETING.csv
[24/53] T_ONTIME_MARKETING.csv
[25/53] T_ONTIME_MARKETING.csv
[26/53] T_ONTIME_MARKETING.csv
[27/53] T_ONTIME_MARKETING.csv
[28/53] T_ONTIME_MARKETING.csv
[29/53] T_ONTIME_MARKETING.csv
[30/53] T_ONTIME_MARKETING.csv
[31/53] T_ONTIME_MARKETING.csv
[32/53] T_ONTIME_MARKETING.csv
[33/53] 

,validation,error_count
0,dep_delay_indicator,0
1,arr_delay_indicator,0
2,cancelled_with_arrival_time,0
3,cancelled_with_air_time,0


### 17.6 Análisis de valores atípicos

Una vez comprobada la integridad estructural y lógica del conjunto de datos, se analiza la presencia de valores atípicos en las principales variables cuantitativas.

La identificación de valores atípicos no implica su eliminación automática. En el contexto del transporte aéreo, valores extremos de retraso, tiempo de taxi o duración pueden corresponder a situaciones operacionales reales.

Por esta razón, el análisis tiene inicialmente carácter descriptivo y busca determinar la magnitud y frecuencia de estos valores antes de tomar decisiones durante la fase de limpieza.

Se analizan las principales variables cuantitativas relacionadas con retrasos y operación del vuelo.

In [34]:
OUTLIER_COLUMNS = [
    "DEP_DELAY",
    "DEP_DELAY_NEW",
    "ARR_DELAY",
    "ARR_DELAY_NEW",
    "TAXI_OUT",
    "TAXI_IN",
    "CRS_ELAPSED_TIME",
    "ACTUAL_ELAPSED_TIME",
    "AIR_TIME",
    "DISTANCE",
    "CARRIER_DELAY",
    "WEATHER_DELAY",
    "NAS_DELAY",
    "SECURITY_DELAY",
    "LATE_AIRCRAFT_DELAY"
]

outlier_stats = {
    column: {
        "count": 0,
        "min": None,
        "max": None,
        "sum": 0.0
    }
    for column in OUTLIER_COLUMNS
}


for file_number, csv_file in enumerate(
    csv_files,
    start=1
):

    print(
        f"[{file_number:02d}/{len(csv_files)}] "
        f"{csv_file.name}"
    )

    reader = pd.read_csv(
        csv_file,
        chunksize=CHUNK_SIZE,
        low_memory=False
    )

    for chunk in reader:

        chunk = normalize_chunk(chunk)

        for column in OUTLIER_COLUMNS:

            values = chunk[column].dropna()

            if values.empty:
                continue

            # Número de observaciones válidas
            outlier_stats[column]["count"] += len(values)

            # Suma
            outlier_stats[column]["sum"] += values.sum()

            # Mínimo
            current_min = values.min()

            if (
                outlier_stats[column]["min"] is None
                or current_min < outlier_stats[column]["min"]
            ):
                outlier_stats[column]["min"] = current_min

            # Máximo
            current_max = values.max()

            if (
                outlier_stats[column]["max"] is None
                or current_max > outlier_stats[column]["max"]
            ):
                outlier_stats[column]["max"] = current_max

[01/53] T_ONTIME_MARKETING.csv
[02/53] T_ONTIME_MARKETING.csv
[03/53] T_ONTIME_MARKETING.csv
[04/53] T_ONTIME_MARKETING.csv
[05/53] T_ONTIME_MARKETING.csv
[06/53] T_ONTIME_MARKETING.csv
[07/53] T_ONTIME_MARKETING.csv
[08/53] T_ONTIME_MARKETING.csv
[09/53] T_ONTIME_MARKETING.csv
[10/53] T_ONTIME_MARKETING.csv
[11/53] T_ONTIME_MARKETING.csv
[12/53] T_ONTIME_MARKETING.csv
[13/53] T_ONTIME_MARKETING.csv
[14/53] T_ONTIME_MARKETING.csv
[15/53] T_ONTIME_MARKETING.csv
[16/53] T_ONTIME_MARKETING.csv
[17/53] T_ONTIME_MARKETING.csv
[18/53] T_ONTIME_MARKETING.csv
[19/53] T_ONTIME_MARKETING.csv
[20/53] T_ONTIME_MARKETING.csv
[21/53] T_ONTIME_MARKETING.csv
[22/53] T_ONTIME_MARKETING.csv
[23/53] T_ONTIME_MARKETING.csv
[24/53] T_ONTIME_MARKETING.csv
[25/53] T_ONTIME_MARKETING.csv
[26/53] T_ONTIME_MARKETING.csv
[27/53] T_ONTIME_MARKETING.csv
[28/53] T_ONTIME_MARKETING.csv
[29/53] T_ONTIME_MARKETING.csv
[30/53] T_ONTIME_MARKETING.csv
[31/53] T_ONTIME_MARKETING.csv
[32/53] T_ONTIME_MARKETING.csv
[33/53] 

In [35]:
# ============================================================
# RESUMEN DE VALORES EXTREMOS
# ============================================================

outlier_summary = []

for column, stats in outlier_stats.items():

    mean = (
        stats["sum"] / stats["count"]
        if stats["count"] > 0
        else None
    )

    outlier_summary.append({
        "variable": column,
        "count": stats["count"],
        "mean": mean,
        "minimum": stats["min"],
        "maximum": stats["max"]
    })


outlier_summary_df = pd.DataFrame(
    outlier_summary
)

display(outlier_summary_df)

,variable,count,mean,minimum,maximum
0,DEP_DELAY,32200169,12.713924,-115.0,7223.0
1,DEP_DELAY_NEW,32200169,16.050677,0.0,7223.0
2,ARR_DELAY,32099041,7.221638,-128.0,7232.0
3,ARR_DELAY_NEW,32099041,16.011519,0.0,7232.0
4,TAXI_OUT,32185094,18.004429,1.0,1274.0
5,TAXI_IN,32173589,8.305459,1.0,1318.0
6,CRS_ELAPSED_TIME,32761119,143.468765,-272.0,1510.0
7,ACTUAL_ELAPSED_TIME,32099032,138.104055,14.0,1354.0
8,AIR_TIME,32099032,111.809039,5.0,1338.0
9,DISTANCE,32761129,803.773069,11.0,5095.0


### 17.6.1 Revisión de valores extremos e inconsistentes

El análisis inicial de las variables cuantitativas muestra valores extremos elevados en diferentes variables relacionadas con retrasos y tiempos operativos.

La presencia de valores extremos no implica necesariamente un error. En el contexto del transporte aéreo pueden producirse retrasos excepcionalmente elevados debido a incidencias operativas, meteorológicas o de infraestructura.

Sin embargo, se detecta un caso que requiere una revisión específica: `CRS_ELAPSED_TIME` presenta valores negativos, llegando hasta -272 minutos.

Dado que esta variable representa la duración programada del vuelo, un valor negativo no es directamente interpretable como una duración física. Por esta razón, se analizarán específicamente los registros afectados antes de tomar cualquier decisión de limpieza.

In [36]:
negative_crs_elapsed_count = 0
negative_crs_elapsed_samples = []

SAMPLE_LIMIT = 100

for file_number, csv_file in enumerate(
    csv_files,
    start=1
):

    print(
        f"[{file_number:02d}/{len(csv_files)}] "
        f"{csv_file.name}"
    )

    reader = pd.read_csv(
        csv_file,
        chunksize=CHUNK_SIZE,
        low_memory=False
    )

    for chunk in reader:

        chunk = normalize_chunk(chunk)

        negative_mask = (
            chunk["CRS_ELAPSED_TIME"] < 0
        )

        negative_rows = chunk.loc[
            negative_mask,
            [
                "FL_DATE",
                "OP_UNIQUE_CARRIER",
                "OP_CARRIER_FL_NUM",
                "ORIGIN",
                "DEST",
                "CRS_DEP_TIME",
                "CRS_ARR_TIME",
                "CRS_ELAPSED_TIME",
                "ACTUAL_ELAPSED_TIME",
                "AIR_TIME",
                "DISTANCE"
            ]
        ]

        negative_crs_elapsed_count += len(
            negative_rows
        )

        if (
            len(negative_crs_elapsed_samples)
            < SAMPLE_LIMIT
            and not negative_rows.empty
        ):

            remaining = (
                SAMPLE_LIMIT
                - len(negative_crs_elapsed_samples)
            )

            negative_crs_elapsed_samples.extend(
                negative_rows
                .head(remaining)
                .to_dict("records")
            )


print(
    f"Registros con CRS_ELAPSED_TIME negativo: "
    f"{negative_crs_elapsed_count:,}"
)

negative_crs_elapsed_df = pd.DataFrame(
    negative_crs_elapsed_samples
)

display(negative_crs_elapsed_df)

[01/53] T_ONTIME_MARKETING.csv
[02/53] T_ONTIME_MARKETING.csv
[03/53] T_ONTIME_MARKETING.csv
[04/53] T_ONTIME_MARKETING.csv
[05/53] T_ONTIME_MARKETING.csv
[06/53] T_ONTIME_MARKETING.csv
[07/53] T_ONTIME_MARKETING.csv
[08/53] T_ONTIME_MARKETING.csv
[09/53] T_ONTIME_MARKETING.csv
[10/53] T_ONTIME_MARKETING.csv
[11/53] T_ONTIME_MARKETING.csv
[12/53] T_ONTIME_MARKETING.csv
[13/53] T_ONTIME_MARKETING.csv
[14/53] T_ONTIME_MARKETING.csv
[15/53] T_ONTIME_MARKETING.csv
[16/53] T_ONTIME_MARKETING.csv
[17/53] T_ONTIME_MARKETING.csv
[18/53] T_ONTIME_MARKETING.csv
[19/53] T_ONTIME_MARKETING.csv
[20/53] T_ONTIME_MARKETING.csv
[21/53] T_ONTIME_MARKETING.csv
[22/53] T_ONTIME_MARKETING.csv
[23/53] T_ONTIME_MARKETING.csv
[24/53] T_ONTIME_MARKETING.csv
[25/53] T_ONTIME_MARKETING.csv
[26/53] T_ONTIME_MARKETING.csv
[27/53] T_ONTIME_MARKETING.csv
[28/53] T_ONTIME_MARKETING.csv
[29/53] T_ONTIME_MARKETING.csv
[30/53] T_ONTIME_MARKETING.csv
[31/53] T_ONTIME_MARKETING.csv
[32/53] T_ONTIME_MARKETING.csv
[33/53] 

,FL_DATE,OP_UNIQUE_CARRIER,OP_CARRIER_FL_NUM,ORIGIN,DEST,CRS_DEP_TIME,CRS_ARR_TIME,CRS_ELAPSED_TIME,ACTUAL_ELAPSED_TIME,AIR_TIME,DISTANCE
0,2026-04-16,WN,697,BNA,MDW,2214,2100,-74.0,NaN,NaN,395.0
1,2026-03-04,WN,3448,SAT,DAL,1950,1825,-85.0,NaN,NaN,247.0
2,2026-03-13,WN,1730,BWI,ROC,12,2355,-17.0,NaN,NaN,277.0
3,2026-03-16,WN,4698,GSP,BNA,2012,1855,-17.0,NaN,NaN,265.0
4,2026-02-17,WN,427,OAK,BUR,2215,2110,-65.0,NaN,NaN,325.0
5,2026-01-01,WN,1067,PHX,LAS,2140,2105,-64.0,NaN,NaN,255.0
6,2026-01-14,WN,373,DAL,SAT,1446,1430,-16.0,NaN,NaN,247.0
7,2026-01-15,WN,1493,STL,DSM,2229,2215,-14.0,NaN,NaN,259.0
8,2026-01-23,WN,1651,BWI,MHT,1254,1210,-44.0,NaN,NaN,377.0
9,2025-11-17,WN,1556,LAS,BUR,1724,1545,-99.0,NaN,NaN,223.0


### 17.6.2 Revisión de retrasos extremos

Las variables de retraso presentan valores máximos superiores a 7.000 minutos.

Aunque estos valores son estadísticamente extremos, no se consideran automáticamente errores. Un retraso excepcional puede reflejar una situación operacional real.

Para comprender su naturaleza se inspeccionan los registros con mayores retrasos de salida y llegada.

In [37]:
top_dep_delays = []
top_arr_delays = []

TOP_N = 20

for csv_file in csv_files:

    reader = pd.read_csv(
        csv_file,
        chunksize=CHUNK_SIZE,
        low_memory=False
    )

    for chunk in reader:

        chunk = normalize_chunk(chunk)

        dep_columns = [
            "FL_DATE",
            "OP_UNIQUE_CARRIER",
            "OP_CARRIER_FL_NUM",
            "ORIGIN",
            "DEST",
            "DEP_DELAY",
            "ARR_DELAY",
            "CANCELLED",
            "DIVERTED"
        ]

        arr_columns = dep_columns

        top_dep_delays.append(
            chunk.nlargest(
                TOP_N,
                "DEP_DELAY"
            )[dep_columns]
        )

        top_arr_delays.append(
            chunk.nlargest(
                TOP_N,
                "ARR_DELAY"
            )[arr_columns]
        )


top_dep_delays_df = (
    pd.concat(top_dep_delays)
    .nlargest(TOP_N, "DEP_DELAY")
    .reset_index(drop=True)
)

top_arr_delays_df = (
    pd.concat(top_arr_delays)
    .nlargest(TOP_N, "ARR_DELAY")
    .reset_index(drop=True)
)

print("Mayores retrasos de salida:")
display(top_dep_delays_df)

print("\nMayores retrasos de llegada:")
display(top_arr_delays_df)

Mayores retrasos de salida:


,FL_DATE,OP_UNIQUE_CARRIER,OP_CARRIER_FL_NUM,ORIGIN,DEST,DEP_DELAY,ARR_DELAY,CANCELLED,DIVERTED
0,2022-05-23,YX,9677,DCA,PWM,7223.0,7232.0,0,0
1,2022-08-07,OO,9663,ORD,RST,5995.0,5986.0,0,0
2,2025-03-19,YX,9678,DCA,DSM,5923.0,5907.0,0,0
3,2023-06-05,YX,9680,DCA,JAX,5764.0,5780.0,0,0
4,2022-06-18,MQ,9631,DFW,MGM,5327.0,5324.0,0,0
5,2023-03-05,AA,810,BOI,DFW,4413.0,4405.0,0,0
6,2025-09-24,AA,797,MSN,DFW,4352.0,4336.0,0,0
7,2025-02-19,PT,9647,MYR,CLT,4340.0,4329.0,0,0
8,2022-05-16,YX,9677,DCA,PWM,4320.0,4318.0,0,0
9,2023-03-03,OH,9660,DFW,SDF,4225.0,4218.0,0,0



Mayores retrasos de llegada:


,FL_DATE,OP_UNIQUE_CARRIER,OP_CARRIER_FL_NUM,ORIGIN,DEST,DEP_DELAY,ARR_DELAY,CANCELLED,DIVERTED
0,2022-05-23,YX,9677,DCA,PWM,7223.0,7232.0,0,0
1,2022-08-07,OO,9663,ORD,RST,5995.0,5986.0,0,0
2,2025-03-19,YX,9678,DCA,DSM,5923.0,5907.0,0,0
3,2023-06-05,YX,9680,DCA,JAX,5764.0,5780.0,0,0
4,2022-06-18,MQ,9631,DFW,MGM,5327.0,5324.0,0,0
5,2023-03-05,AA,810,BOI,DFW,4413.0,4405.0,0,0
6,2025-09-24,AA,797,MSN,DFW,4352.0,4336.0,0,0
7,2025-02-19,PT,9647,MYR,CLT,4340.0,4329.0,0,0
8,2022-05-16,YX,9677,DCA,PWM,4320.0,4318.0,0,0
9,2023-03-03,OH,9660,DFW,SDF,4225.0,4218.0,0,0


In [38]:
print("\nMayores retrasos de llegada:")
display(top_arr_delays_df)


Mayores retrasos de llegada:


,FL_DATE,OP_UNIQUE_CARRIER,OP_CARRIER_FL_NUM,ORIGIN,DEST,DEP_DELAY,ARR_DELAY,CANCELLED,DIVERTED
0,2022-05-23,YX,9677,DCA,PWM,7223.0,7232.0,0,0
1,2022-08-07,OO,9663,ORD,RST,5995.0,5986.0,0,0
2,2025-03-19,YX,9678,DCA,DSM,5923.0,5907.0,0,0
3,2023-06-05,YX,9680,DCA,JAX,5764.0,5780.0,0,0
4,2022-06-18,MQ,9631,DFW,MGM,5327.0,5324.0,0,0
5,2023-03-05,AA,810,BOI,DFW,4413.0,4405.0,0,0
6,2025-09-24,AA,797,MSN,DFW,4352.0,4336.0,0,0
7,2025-02-19,PT,9647,MYR,CLT,4340.0,4329.0,0,0
8,2022-05-16,YX,9677,DCA,PWM,4320.0,4318.0,0,0
9,2023-03-03,OH,9660,DFW,SDF,4225.0,4218.0,0,0


### 17.6.3 Evaluación de valores negativos en `CRS_ELAPSED_TIME`

Se identificaron 21 registros con valores negativos en la variable `CRS_ELAPSED_TIME`.

Esta variable representa la duración programada del vuelo en minutos, por lo que los valores negativos no son físicamente interpretables como una duración válida.

Los registros afectados representan una proporción extremadamente reducida respecto al conjunto completo de datos. Además, en algunos casos existen valores positivos y coherentes de `ACTUAL_ELAPSED_TIME` y `AIR_TIME`, lo que indica que el problema se concentra en la información programada del vuelo.

Debido a que no existe una regla suficientemente robusta para reconstruir el valor programado original sin introducir información artificial, estos registros no serán corregidos manualmente.

La decisión metodológica será conservar los registros en el dataset maestro para mantener la trazabilidad de la fuente, pero considerar `CRS_ELAPSED_TIME` como inválido en estos casos y excluir dichas observaciones de los análisis o modelos que requieran esta variable.

### 17.7 Resumen del análisis de calidad de los datos

El análisis de calidad se realizó sobre un total de 32.761.129 registros, evaluando valores faltantes, duplicados, rangos válidos, consistencia lógica y valores extremos.

Los resultados muestran una elevada consistencia estructural del conjunto de datos. No se detectaron registros duplicados según la clave operacional definida, ni valores fuera de los dominios establecidos para las variables binarias, temporales y principales magnitudes operacionales.

El análisis de valores faltantes permitió comprobar que una parte importante de estos valores tiene carácter estructural y responde a la propia lógica operacional del dataset. Por ejemplo, `CANCELLATION_CODE` únicamente se encuentra informado para vuelos cancelados, mientras que las variables correspondientes a las causas del retraso se encuentran principalmente informadas cuando existe un retraso de llegada igual o superior a 15 minutos.

Se detectaron únicamente tres vuelos con retraso de llegada igual o superior a 15 minutos sin información en las variables de causa del retraso. Estos registros serán considerados casos particulares durante la preparación definitiva de los datos.

En las variables operacionales se identificaron valores extremos, especialmente en los retrasos de salida y llegada. Sin embargo, la comparación entre ambas variables muestra que los mayores retrasos presentan valores consistentes entre salida y llegada. Por esta razón, estos valores no se consideran automáticamente errores, sino observaciones extremas potencialmente reales que deben conservarse inicialmente.

Finalmente, se identificaron 21 registros con valores negativos en `CRS_ELAPSED_TIME`. Debido a que esta variable representa la duración programada del vuelo, dichos valores no tienen una interpretación física válida. Al representar una proporción extremadamente reducida del conjunto de datos, estos casos serán tratados específicamente durante la fase de limpieza sin eliminar necesariamente el registro completo.

En conjunto, los resultados permiten concluir que el dataset presenta una calidad estructural elevada y que los principales tratamientos necesarios están relacionados con la interpretación de valores faltantes estructurales y un número muy reducido de inconsistencias específicas.

In [39]:
# ------------------------------------------------------------
# 1. Recuperar métricas calculadas anteriormente
# ------------------------------------------------------------

cancellation_null_pct = null_profile_df.loc[
    null_profile_df["variable"] == "CANCELLATION_CODE",
    "null_percentage"
].iloc[0]


delay_causes_null_pct = null_profile_df.loc[
    null_profile_df["variable"] == "CARRIER_DELAY",
    "null_percentage"
].iloc[0]


delayed_without_causes = (
    missing_context["arr_delayed_15_without_causes"]
)


total_range_errors = sum(
    range_errors.values()
)


total_consistency_errors = sum(
    consistency_errors.values()
)


# ------------------------------------------------------------
# 2. Obtener máximo retraso observado
# ------------------------------------------------------------

max_dep_delay = outlier_summary_df.loc[
    outlier_summary_df["variable"] == "DEP_DELAY",
    "maximum"
].iloc[0]

max_arr_delay = outlier_summary_df.loc[
    outlier_summary_df["variable"] == "ARR_DELAY",
    "maximum"
].iloc[0]

max_delay = max(
    max_dep_delay,
    max_arr_delay
)


# ------------------------------------------------------------
# 3. Calcular proporción de CRS_ELAPSED_TIME inválido
# ------------------------------------------------------------

negative_crs_elapsed_pct = (
    negative_crs_elapsed_count
    / total_rows
    * 100
)


# ------------------------------------------------------------
# 4. Construcción dinámica del resumen
# ------------------------------------------------------------

quality_summary = pd.DataFrame([

    {
        "hallazgo": "Registros analizados",
        "resultado": f"{total_rows:,}",
        "interpretacion":
            "Número total de observaciones evaluadas",
        "decision":
            "Base del análisis de calidad"
    },

    {
        "hallazgo": "Registros duplicados",
        "resultado": f"{duplicate_count:,}",
        "interpretacion":
            "Duplicados detectados mediante la clave operacional",
        "decision":
            (
                "No requiere tratamiento"
                if duplicate_count == 0
                else "Revisar y tratar duplicados"
            )
    },

    {
        "hallazgo": "Valores fuera de rango",
        "resultado": f"{total_range_errors:,}",
        "interpretacion":
            "Incumplimientos de los dominios definidos",
        "decision":
            (
                "No requiere tratamiento"
                if total_range_errors == 0
                else "Revisar registros afectados"
            )
    },

    {
        "hallazgo": "Inconsistencias lógicas",
        "resultado": f"{total_consistency_errors:,}",
        "interpretacion":
            "Inconsistencias entre variables relacionadas",
        "decision":
            (
                "No requiere tratamiento"
                if total_consistency_errors == 0
                else "Revisar registros afectados"
            )
    },

    {
        "hallazgo": "CANCELLATION_CODE nulo",
        "resultado": f"{cancellation_null_pct:.2f}%",
        "interpretacion":
            "Nulo estructural principalmente asociado a vuelos no cancelados",
        "decision":
            "Conservar"
    },

    {
        "hallazgo": "Variables de causa del retraso nulas",
        "resultado": f"{delay_causes_null_pct:.2f}%",
        "interpretacion":
            "Valores mayoritariamente estructurales según el estado del vuelo",
        "decision":
            "Conservar y tratar según el objetivo analítico"
    },

    {
        "hallazgo": "Vuelos retrasados sin causa informada",
        "resultado": f"{delayed_without_causes:,}",
        "interpretacion":
            "Vuelos con ARR_DEL15 = 1 sin causa de retraso informada",
        "decision":
            (
                "No requiere tratamiento"
                if delayed_without_causes == 0
                else "Revisar durante la preparación de datos"
            )
    },

    {
        "hallazgo": "CRS_ELAPSED_TIME negativo",
        "resultado": (
            f"{negative_crs_elapsed_count:,} "
            f"({negative_crs_elapsed_pct:.6f}%)"
        ),
        "interpretacion":
            "Duraciones programadas con valores físicamente inválidos",
        "decision":
            "Invalidar el valor sin eliminar inicialmente el vuelo"
    },

    {
        "hallazgo": "Máximo retraso observado",
        "resultado": f"{max_delay:,.0f} minutos",
        "interpretacion":
            "Valor extremo detectado en los retrasos de salida o llegada",
        "decision":
            "Conservar inicialmente como observación operacional"
    }

])


# ------------------------------------------------------------
# 5. Mostrar resumen
# ------------------------------------------------------------

display(quality_summary)

,hallazgo,resultado,interpretacion,decision
0,Registros analizados,"32,761,129",Número total de observaciones evaluadas,Base del análisis de calidad
1,Registros duplicados,0,Duplicados detectados mediante la clave operac...,No requiere tratamiento
2,Valores fuera de rango,0,Incumplimientos de los dominios definidos,No requiere tratamiento
3,Inconsistencias lógicas,0,Inconsistencias entre variables relacionadas,No requiere tratamiento
4,CANCELLATION_CODE nulo,98.23%,Nulo estructural principalmente asociado a vue...,Conservar
5,Variables de causa del retraso nulas,79.35%,Valores mayoritariamente estructurales según e...,Conservar y tratar según el objetivo analítico
6,Vuelos retrasados sin causa informada,3,Vuelos con ARR_DEL15 = 1 sin causa de retraso ...,Revisar durante la preparación de datos
7,CRS_ELAPSED_TIME negativo,21 (0.000064%),Duraciones programadas con valores físicamente...,Invalidar el valor sin eliminar inicialmente e...
8,Máximo retraso observado,"7,232 minutos",Valor extremo detectado en los retrasos de sal...,Conservar inicialmente como observación operac...


## 18. Limpieza de los datos

Una vez completado el análisis de calidad, se procede a definir las reglas de limpieza que serán aplicadas al conjunto de datos.

A diferencia de la normalización estructural realizada en el apartado 16, cuyo objetivo era homogeneizar formatos y tipos de datos, esta fase incorpora decisiones metodológicas derivadas directamente de los problemas identificados durante el análisis de calidad.

Las reglas de limpieza se definen antes de implementar la función que las aplicará, con el objetivo de separar claramente la identificación de problemas de las decisiones adoptadas para resolverlos.

### 18.1 Definición de las reglas de limpieza

A partir de los resultados obtenidos en el apartado 17 se establecen las siguientes decisiones:

1. **Registros duplicados**

   No se detectaron duplicados mediante la clave operacional definida. Por tanto, no será necesario eliminar registros por este motivo.

2. **Valores faltantes estructurales**

   Los valores ausentes de `CANCELLATION_CODE` y de las variables correspondientes a las causas del retraso responden principalmente a la lógica operacional del conjunto de datos.

   Estos valores no serán eliminados ni imputados de forma general, ya que la ausencia de información tiene significado dentro del contexto del vuelo.

3. **Valores negativos de `CRS_ELAPSED_TIME`**

   Se detectaron valores negativos en esta variable. Al representar una duración programada, dichos valores no son físicamente interpretables.

   En lugar de eliminar el vuelo completo, el valor de `CRS_ELAPSED_TIME` será sustituido por un valor ausente (`NaN`), manteniendo el resto de la información del registro.

4. **Vuelos retrasados sin causa informada**

   Se identificó un número muy reducido de vuelos con `ARR_DEL15 = 1` sin información en ninguna de las variables correspondientes a las causas del retraso.

   Estos valores no serán imputados artificialmente, ya que no existe información suficiente para determinar la causa real. Se conservarán como valores ausentes.

5. **Valores extremos**

   Se identificaron valores extremos en diferentes variables operacionales, especialmente en los retrasos de salida y llegada.

   La consistencia observada entre los retrasos de salida y llegada indica que estos valores pueden representar situaciones operacionales reales. Por tanto, no serán eliminados ni recortados durante la limpieza general.

Estas reglas permiten conservar la máxima cantidad posible de información original y evitar introducir valores artificiales sin una justificación basada en los datos.

### 18.2 Validación de `clean_chunk()`

`clean_chunk()` se encuentra implementada en `src.preprocessing`. En este bloque no se
redefine la función: únicamente se comprueba que la regla de limpieza validada en el
análisis de calidad se comporte como se espera.

Esta separación es importante: el notebook demuestra la decisión metodológica y el
módulo Python contiene la implementación reutilizable.


In [40]:
validation_df = pd.DataFrame({
    "CRS_ELAPSED_TIME": [
        120.0,
        -74.0,
        95.0,
        -17.0
    ],
    "DISTANCE": [
        500.0,
        395.0,
        300.0,
        277.0
    ]
})

# Utilizamos Float32 para mantener el tipo definido
# en la normalización del proyecto.

validation_df["CRS_ELAPSED_TIME"] = (
    validation_df["CRS_ELAPSED_TIME"]
    .astype("Float32")
)

validation_df["DISTANCE"] = (
    validation_df["DISTANCE"]
    .astype("Float32")
)


print("ANTES DE LA LIMPIEZA:")
display(validation_df)


validation_clean_df = clean_chunk(
    validation_df
)


print("DESPUÉS DE LA LIMPIEZA:")
display(validation_clean_df)


print(
    "Registros antes:",
    len(validation_df)
)

print(
    "Registros después:",
    len(validation_clean_df)
)

print(
    "CRS_ELAPSED_TIME negativos restantes:",
    (
        validation_clean_df["CRS_ELAPSED_TIME"] < 0
    ).sum()
)

ANTES DE LA LIMPIEZA:


,CRS_ELAPSED_TIME,DISTANCE
0,120.0,500.0
1,-74.0,395.0
2,95.0,300.0
3,-17.0,277.0


DESPUÉS DE LA LIMPIEZA:


,CRS_ELAPSED_TIME,DISTANCE
0,120.0,500.0
1,<NA>,395.0
2,95.0,300.0
3,<NA>,277.0


Registros antes: 4
Registros después: 4
CRS_ELAPSED_TIME negativos restantes: 0


### 18.3 Prueba del proceso completo sobre datos reales

Una vez validada de forma controlada la función de limpieza, se realiza una prueba sobre datos reales.

El procesamiento se ejecuta de forma secuencial:

1. Lectura del archivo original.
2. Normalización estructural mediante `normalize_chunk()`.
3. Limpieza mediante `clean_chunk()`.

Esta secuencia representa la estructura que posteriormente será utilizada para procesar el conjunto completo de archivos.

In [41]:
test_file = csv_files[0]

test_raw_df = pd.read_csv(
    test_file,
    nrows=100_000,
    low_memory=False
)

# Normalización
test_normalized_df = normalize_chunk(
    test_raw_df
)

# Limpieza
test_clean_df = clean_chunk(
    test_normalized_df
)


print("Archivo:")
print(test_file)

print("\nRegistros originales:")
print(len(test_raw_df))

print("\nRegistros normalizados:")
print(len(test_normalized_df))

print("\nRegistros después de limpieza:")
print(len(test_clean_df))

print("\nCRS_ELAPSED_TIME negativos restantes:")
print(
    (
        test_clean_df["CRS_ELAPSED_TIME"] < 0
    ).sum()
)

Archivo:
G:\My Drive\MASTER Big Data\TFM\data\raw\Flights\T_ONTIME_MARKETING_20260807_125204\T_ONTIME_MARKETING.csv

Registros originales:
100000

Registros normalizados:
100000

Registros después de limpieza:
100000

CRS_ELAPSED_TIME negativos restantes:
0


### 18.4 Validación global de la limpieza

Una vez validado el funcionamiento de las funciones `normalize_chunk()` y `clean_chunk()` sobre una muestra de datos reales, se procede a comprobar su comportamiento sobre el conjunto completo.

El procesamiento se realiza por bloques para evitar cargar simultáneamente los más de 32 millones de registros en memoria.

La validación global tiene como objetivos:

- Comprobar que todos los registros pueden ser procesados sin errores.
- Verificar que la limpieza no elimina observaciones.
- Confirmar que no permanecen valores negativos en `CRS_ELAPSED_TIME`.
- Cuantificar el número de valores modificados durante la limpieza.
- Garantizar que todos los archivos producen exactamente las 45 variables definidas para el dataset analítico.

Esta fase constituye la validación final del pipeline de preparación antes de generar los archivos procesados definitivos.

In [42]:
CHUNK_SIZE = 100_000

total_raw_rows = 0
total_clean_rows = 0
total_corrected_crs_elapsed = 0
remaining_negative_crs_elapsed = 0

files_processed = 0
files_with_errors = []

for file_number, csv_file in enumerate(
    csv_files,
    start=1
):

    print(
        f"[{file_number:02d}/{len(csv_files)}] "
        f"{csv_file.name}"
    )

    try:

        reader = pd.read_csv(
            csv_file,
            chunksize=CHUNK_SIZE,
            low_memory=False
        )

        file_raw_rows = 0
        file_clean_rows = 0

        for chunk in reader:

            # -----------------------------------------------
            # Registros originales
            # -----------------------------------------------
            raw_rows = len(chunk)

            total_raw_rows += raw_rows
            file_raw_rows += raw_rows

            # -----------------------------------------------
            # Normalización
            # -----------------------------------------------
            chunk = normalize_chunk(chunk)

            # -----------------------------------------------
            # Contar valores que serán corregidos
            # -----------------------------------------------
            invalid_crs_mask = (
                chunk["CRS_ELAPSED_TIME"] < 0
            )

            corrected_in_chunk = int(
                invalid_crs_mask.sum()
            )

            total_corrected_crs_elapsed += (
                corrected_in_chunk
            )

            # -----------------------------------------------
            # Limpieza
            # -----------------------------------------------
            chunk = clean_chunk(chunk)

            # -----------------------------------------------
            # Registros posteriores a limpieza
            # -----------------------------------------------
            clean_rows = len(chunk)

            total_clean_rows += clean_rows
            file_clean_rows += clean_rows

            # -----------------------------------------------
            # Comprobar negativos restantes
            # -----------------------------------------------
            remaining_negative_crs_elapsed += int(
                (
                    chunk["CRS_ELAPSED_TIME"] < 0
                ).sum()
            )

            # -----------------------------------------------
            # Validar número de columnas
            # -----------------------------------------------
            if len(chunk.columns) != len(
                FINAL_COLUMN_ORDER
            ):
                raise ValueError(
                    "Número de columnas inesperado: "
                    f"{len(chunk.columns)}"
                )

        # -----------------------------------------------
        # Validación del archivo
        # -----------------------------------------------
        if file_raw_rows != file_clean_rows:
            raise ValueError(
                "El número de registros cambió durante "
                "la limpieza."
            )

        files_processed += 1

    except Exception as error:

        files_with_errors.append({
            "file": csv_file.name,
            "path": str(csv_file),
            "error": str(error)
        })

        print(
            f"ERROR procesando {csv_file.name}: "
            f"{error}"
        )


# ============================================================
# RESULTADOS
# ============================================================

print("\n" + "=" * 60)
print("VALIDACIÓN GLOBAL")
print("=" * 60)

print(
    f"Archivos procesados correctamente : "
    f"{files_processed:,}"
)

print(
    f"Archivos con errores              : "
    f"{len(files_with_errors):,}"
)

print(
    f"Registros originales              : "
    f"{total_raw_rows:,}"
)

print(
    f"Registros después de limpieza     : "
    f"{total_clean_rows:,}"
)

print(
    f"CRS_ELAPSED_TIME corregidos       : "
    f"{total_corrected_crs_elapsed:,}"
)

print(
    f"CRS_ELAPSED_TIME negativos finales: "
    f"{remaining_negative_crs_elapsed:,}"
)

[01/53] T_ONTIME_MARKETING.csv
[02/53] T_ONTIME_MARKETING.csv
[03/53] T_ONTIME_MARKETING.csv
[04/53] T_ONTIME_MARKETING.csv
[05/53] T_ONTIME_MARKETING.csv
[06/53] T_ONTIME_MARKETING.csv
[07/53] T_ONTIME_MARKETING.csv
[08/53] T_ONTIME_MARKETING.csv
[09/53] T_ONTIME_MARKETING.csv
[10/53] T_ONTIME_MARKETING.csv
[11/53] T_ONTIME_MARKETING.csv
[12/53] T_ONTIME_MARKETING.csv
[13/53] T_ONTIME_MARKETING.csv
[14/53] T_ONTIME_MARKETING.csv
[15/53] T_ONTIME_MARKETING.csv
[16/53] T_ONTIME_MARKETING.csv
[17/53] T_ONTIME_MARKETING.csv
[18/53] T_ONTIME_MARKETING.csv
[19/53] T_ONTIME_MARKETING.csv
[20/53] T_ONTIME_MARKETING.csv
[21/53] T_ONTIME_MARKETING.csv
[22/53] T_ONTIME_MARKETING.csv
[23/53] T_ONTIME_MARKETING.csv
[24/53] T_ONTIME_MARKETING.csv
[25/53] T_ONTIME_MARKETING.csv
[26/53] T_ONTIME_MARKETING.csv
[27/53] T_ONTIME_MARKETING.csv
[28/53] T_ONTIME_MARKETING.csv
[29/53] T_ONTIME_MARKETING.csv
[30/53] T_ONTIME_MARKETING.csv
[31/53] T_ONTIME_MARKETING.csv
[32/53] T_ONTIME_MARKETING.csv
[33/53] 

### 18.5 Resumen final de la limpieza

La validación global del proceso de limpieza se realizó sobre la totalidad de los archivos identificados en la fuente de datos.

El pipeline procesó correctamente todos los archivos sin generar errores y sin eliminar registros. El número de observaciones antes y después de la limpieza se mantuvo constante, confirmando que las reglas implementadas preservan la información original siempre que no exista una justificación objetiva para su eliminación.

Los valores negativos detectados previamente en `CRS_ELAPSED_TIME` fueron correctamente invalidados, sin eliminar los vuelos a los que pertenecían. Tras la limpieza no permanecen valores negativos en esta variable.

Los valores faltantes estructurales se conservaron, dado que su ausencia responde a la lógica operacional del conjunto de datos. Del mismo modo, los valores extremos de retraso no fueron eliminados al existir evidencia de consistencia entre las variables de salida y llegada.

Por tanto, la fase de limpieza se considera completada y el conjunto de datos está preparado para la siguiente etapa de transformación e ingeniería de características.

## 19. Generación de la capa procesada

Una vez cerradas las reglas de normalización y limpieza, se genera la primera carga
histórica en Parquet.

> **Nota de operación:** esta sección corresponde a la construcción inicial de la capa
> procesada. En futuras actualizaciones rutinarias no debe reprocesarse el histórico;
> para ello se utiliza el mecanismo incremental del bloque 20.


### 19.1 Definición de los directorios de salida

La capa procesada se almacenará dentro de la carpeta:

`data/processed/flights`

Cada archivo CSV original generará uno o varios archivos Parquet procesados.

Se mantendrá una estructura separada de los datos originales para garantizar la trazabilidad y evitar cualquier modificación accidental de la fuente.


In [43]:
PROCESSED_FLIGHTS_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "flights"
)

PROCESSED_FLIGHTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Directorio de salida:")
print(PROCESSED_FLIGHTS_DIR)

print("\nExiste:")
print(PROCESSED_FLIGHTS_DIR.exists())

Directorio de salida:
G:\My Drive\MASTER Big Data\TFM\data\processed\flights

Existe:
True


### 19.2 Estrategia de particionado

Debido al volumen del dataset, no se almacenará toda la información en un único archivo Parquet.

La capa procesada se organizará mediante particiones por año y mes.

Esta estrategia permite:

- Reducir el tamaño de cada archivo.
- Leer únicamente los periodos necesarios.
- Facilitar el análisis temporal.
- Mejorar el rendimiento en fases posteriores.
- Mantener una estructura clara y escalable.

La estructura resultante será similar a:

`data/processed/flights/YEAR=2025/MONTH=1/...`


### 19.3 Procesamiento definitivo de los archivos

Se procesan todos los archivos CSV mediante lectura por bloques.

Cada bloque sigue el pipeline definido anteriormente:

1. Lectura desde CSV.
2. Normalización mediante `normalize_chunk()`.
3. Limpieza mediante `clean_chunk()`.
4. Escritura en Parquet.

La lectura por bloques permite mantener controlado el uso de memoria independientemente del volumen total del dataset.


In [44]:
# IMPORTANTE:
# Este bloque construye la primera versión completa de la capa procesada.
# Una vez creado el ingestion_manifest, las actualizaciones posteriores
# deben realizarse mediante el bloque 20 para evitar reprocesar el histórico.

CHUNK_SIZE = 100_000

processed_rows = 0
processed_chunks = 0
processed_files = 0
processing_errors = []

for file_number, csv_file in enumerate(csv_files, start=1):
    print(f"[{file_number:02d}/{len(csv_files)}] {csv_file.name}")

    try:
        # La lectura por chunks limita el consumo de memoria.
        reader = pd.read_csv(
            csv_file,
            chunksize=CHUNK_SIZE,
            low_memory=False,
        )

        file_chunks = 0
        file_rows = 0

        for chunk_number, chunk in enumerate(reader, start=1):
            # 1) Contrato estructural común del proyecto.
            chunk = normalize_chunk(chunk)

            # 2) Reglas de calidad ya justificadas en el bloque 17.
            chunk = clean_chunk(chunk)

            # 3) Particionar por año/mes permite leer solo el periodo
            #    necesario en EDA, series temporales o actualizaciones.
            for (year, month), group_df in chunk.groupby(
                ["YEAR", "MONTH"],
                dropna=False,
            ):
                year_dir = PROCESSED_FLIGHTS_DIR / f"YEAR={int(year)}"
                month_dir = year_dir / f"MONTH={int(month):02d}"
                month_dir.mkdir(parents=True, exist_ok=True)

                output_file = (
                    month_dir
                    / f"{csv_file.parent.name}_chunk_{chunk_number:05d}.parquet"
                )

                group_df.to_parquet(
                    output_file,
                    engine="pyarrow",
                    compression="snappy",
                    index=False,
                )

            file_rows += len(chunk)
            processed_rows += len(chunk)
            file_chunks += 1
            processed_chunks += 1

        processed_files += 1
        print(f"   Filas procesadas: {file_rows:,}")

    except Exception as error:
        processing_errors.append(
            {
                "file": csv_file.name,
                "path": str(csv_file),
                "error": str(error),
            }
        )
        print(f"   ERROR: {error}")

print("\n" + "=" * 60)
print("PROCESAMIENTO FINALIZADO")
print("=" * 60)
print(f"Archivos procesados : {processed_files:,}")
print(f"Chunks procesados   : {processed_chunks:,}")
print(f"Registros procesados: {processed_rows:,}")
print(f"Errores              : {len(processing_errors):,}")


[01/53] T_ONTIME_MARKETING.csv
   Filas procesadas: 677,208
[02/53] T_ONTIME_MARKETING.csv
   Filas procesadas: 660,674
[03/53] T_ONTIME_MARKETING.csv
   Filas procesadas: 675,500
[04/53] T_ONTIME_MARKETING.csv
   Filas procesadas: 568,789
[05/53] T_ONTIME_MARKETING.csv
   Filas procesadas: 602,953
[06/53] T_ONTIME_MARKETING.csv
   Filas procesadas: 599,013
[07/53] T_ONTIME_MARKETING.csv
   Filas procesadas: 559,577
[08/53] T_ONTIME_MARKETING.csv
   Filas procesadas: 664,932
[09/53] T_ONTIME_MARKETING.csv
   Filas procesadas: 644,084
[10/53] T_ONTIME_MARKETING.csv
   Filas procesadas: 667,586
[11/53] T_ONTIME_MARKETING.csv
   Filas procesadas: 674,179
[12/53] T_ONTIME_MARKETING.csv
   Filas procesadas: 696,049
[13/53] T_ONTIME_MARKETING.csv
   Filas procesadas: 666,242
[14/53] T_ONTIME_MARKETING.csv
   Filas procesadas: 621,601
[15/53] T_ONTIME_MARKETING.csv
   Filas procesadas: 668,332
[16/53] T_ONTIME_MARKETING.csv
   Filas procesadas: 630,188
[17/53] T_ONTIME_MARKETING.csv
   Filas 

### 19.4 Validación de la capa procesada

Una vez generados los archivos Parquet, se verifica que la capa procesada haya sido creada correctamente.

Se comprueba:

- Número total de archivos Parquet generados.
- Estructura de las particiones.
- Número total de registros almacenados.
- Correspondencia con el número de registros procesados.
- Se inspecciona el esquema de uno de los archivos Parquet para comprobar que los tipos definidos durante la normalización se han conservado correctamen


In [45]:
parquet_files = sorted(
    PROCESSED_FLIGHTS_DIR.rglob(
        "*.parquet"
    )
)

print(
    f"Archivos Parquet generados: "
    f"{len(parquet_files):,}"
)

print("\nPrimeros archivos:")

for parquet_file in parquet_files[:10]:
    print(parquet_file)


sample_parquet = parquet_files[0]

sample_processed_df = pd.read_parquet(
    sample_parquet
)

print("\n\n\nArchivo:")
print(sample_parquet)

print("\nDimensiones:")
print(sample_processed_df.shape)

print("\nTipos:")
display(
    pd.DataFrame({
        "variable": sample_processed_df.columns,
        "dtype": (
            sample_processed_df
            .dtypes
            .astype(str)
            .values
        )
    })
)

Archivos Parquet generados: 354

Primeros archivos:
G:\My Drive\MASTER Big Data\TFM\data\processed\flights\YEAR=2022\MONTH=01\T_ONTIME_MARKETING_20260807_145601_chunk_00001.parquet
G:\My Drive\MASTER Big Data\TFM\data\processed\flights\YEAR=2022\MONTH=01\T_ONTIME_MARKETING_20260807_145601_chunk_00002.parquet
G:\My Drive\MASTER Big Data\TFM\data\processed\flights\YEAR=2022\MONTH=01\T_ONTIME_MARKETING_20260807_145601_chunk_00003.parquet
G:\My Drive\MASTER Big Data\TFM\data\processed\flights\YEAR=2022\MONTH=01\T_ONTIME_MARKETING_20260807_145601_chunk_00004.parquet
G:\My Drive\MASTER Big Data\TFM\data\processed\flights\YEAR=2022\MONTH=01\T_ONTIME_MARKETING_20260807_145601_chunk_00005.parquet
G:\My Drive\MASTER Big Data\TFM\data\processed\flights\YEAR=2022\MONTH=01\T_ONTIME_MARKETING_20260807_145601_chunk_00006.parquet
G:\My Drive\MASTER Big Data\TFM\data\processed\flights\YEAR=2022\MONTH=02\T_ONTIME_MARKETING_20260807_145715_chunk_00001.parquet
G:\My Drive\MASTER Big Data\TFM\data\processe

,variable,dtype
0,FL_DATE,datetime64[ns]
1,YEAR,Int16
2,QUARTER,Int8
3,MONTH,Int8
4,DAY_OF_MONTH,Int8
5,DAY_OF_WEEK,Int8
6,MKT_UNIQUE_CARRIER,string
7,OP_UNIQUE_CARRIER,string
8,TAIL_NUM,string
9,OP_CARRIER_FL_NUM,Int32


In [46]:
import pyarrow.parquet as pq


total_parquet_rows = 0

for parquet_file in parquet_files:

    metadata = pq.ParquetFile(
        parquet_file
    ).metadata

    total_parquet_rows += (
        metadata.num_rows
    )


print(
    f"Registros procesados : "
    f"{processed_rows:,}"
)

print(
    f"Registros en Parquet : "
    f"{total_parquet_rows:,}"
)

print(
    f"Diferencia           : "
    f"{processed_rows - total_parquet_rows:,}"
)

Registros procesados : 32,761,129
Registros en Parquet : 32,761,129
Diferencia           : 0


### 19.5 Generación de metadatos de la capa procesada

Se genera un archivo resumen con información sobre la capa procesada.

Este archivo documenta:

- Número de archivos originales procesados.
- Número total de registros.
- Número de archivos Parquet generados.
- Número de variables.
- Periodo temporal cubierto.
- Número de errores producidos durante el procesamiento.

Estos metadatos facilitarán la trazabilidad y documentación posterior del proyecto.


In [47]:
min_date = None
max_date = None

for parquet_file in parquet_files:

    dates = pd.read_parquet(
        parquet_file,
        columns=["FL_DATE"]
    )["FL_DATE"]

    current_min = dates.min()
    current_max = dates.max()

    if (
        min_date is None
        or current_min < min_date
    ):
        min_date = current_min

    if (
        max_date is None
        or current_max > max_date
    ):
        max_date = current_max


processed_metadata = pd.DataFrame([
    {
        "raw_files": len(csv_files),
        "processed_files": processed_files,
        "records": total_parquet_rows,
        "variables": len(FINAL_COLUMN_ORDER),
        "parquet_files": len(parquet_files),
        "min_date": min_date,
        "max_date": max_date,
        "processing_errors": len(
            processing_errors
        )
    }
])

display(processed_metadata)

metadata_file = (
    PROCESSED_FLIGHTS_DIR
    / "dataset_metadata.csv"
)

processed_metadata.to_csv(
    metadata_file,
    index=False
)

print(
    "Metadatos guardados en:"
)

print(metadata_file)

,raw_files,processed_files,records,variables,parquet_files,min_date,max_date,processing_errors
0,53,53,32761129,45,354,2022-01-01,2026-05-31,0


Metadatos guardados en:
G:\My Drive\MASTER Big Data\TFM\data\processed\flights\dataset_metadata.csv


## 20. Actualización incremental de datos

A partir de este bloque comienza el flujo **productivo y reutilizable** de la ingesta.

La lógica cambia respecto de la carga histórica inicial: antes de transformar datos se
consulta `ingestion_manifest.csv`. Solo los archivos cuyo contenido aún no esté
registrado continúan hacia validación, normalización, limpieza y escritura.

Flujo operativo:

`raw → manifest → detectar pendientes → validar → normalizar → limpiar → Parquet → actualizar manifest`

De esta manera, una nueva publicación mensual no obliga a procesar nuevamente todo el
histórico.


In [48]:
from datetime import datetime
import pyarrow.parquet as pq

# Archivos de control de la capa procesada.
MANIFEST_FILE = PROCESSED_FLIGHTS_DIR / "ingestion_manifest.csv"
METADATA_FILE = PROCESSED_FLIGHTS_DIR / "dataset_metadata.csv"

# El mismo tamaño de chunk se utiliza para mantener un comportamiento
# consistente entre la carga histórica y las actualizaciones.
CHUNK_SIZE = 100_000


### 20.2 Carga del manifest de ingesta

El archivo `ingestion_manifest.csv` mantiene el historial de fuentes incorporadas a la capa procesada.

Si el manifest existe, se carga su contenido para identificar los archivos ya procesados.

Si todavía no existe, se crea una estructura vacía que permitirá registrar la primera ejecución del pipeline.

In [49]:
# Esta llamada es el primer control del pipeline productivo.
# Si el manifest existe, recupera el histórico de fuentes ya cargadas.
# Si no existe, devuelve una estructura vacía preparada para la primera carga.
ingestion_manifest = load_ingestion_manifest(MANIFEST_FILE)

if ingestion_manifest.empty:
    print("Manifest vacío o inexistente: no hay fuentes registradas.")
else:
    print(
        f"Manifest encontrado: {len(ingestion_manifest):,} "
        "archivos registrados."
    )


Manifest encontrado: 53 archivos registrados.


### 20.3 Identificación de archivos pendientes

Se vuelve a inspeccionar la carpeta de datos originales para identificar todos los CSV actualmente disponibles.

Cada archivo se compara con el manifest mediante su hash SHA-256.

El uso del hash permite identificar el contenido real del archivo y evita depender únicamente del nombre, que puede cambiar entre descargas.

Los archivos cuyo hash ya figure con estado `PROCESSED` son omitidos. Únicamente los archivos nuevos continúan hacia las siguientes fases del pipeline.

In [50]:
# 1) Volver a descubrir las fuentes permite detectar archivos añadidos
#    después de la ejecución anterior.
csv_files_current = discover_csv_files(FLIGHTS_DIR)

# 2) SHA-256 se compara con el manifest.
#    Solo new_files seguirá hacia las fases costosas del pipeline.
new_files, already_processed_files = detect_new_files(
    csv_files_current,
    ingestion_manifest,
)

print("\n" + "=" * 60)
print("ESTADO DEL PIPELINE")
print("=" * 60)
print(f"Archivos disponibles : {len(csv_files_current):,}")
print(f"Ya procesados        : {len(already_processed_files):,}")
print(f"Pendientes           : {len(new_files):,}")



ESTADO DEL PIPELINE
Archivos disponibles : 53
Ya procesados        : 53
Pendientes           : 0


#### Control de ejecución incremental

Si no se detectan nuevos archivos, el pipeline no ejecuta las fases de transformación y escritura.

Esto evita operaciones innecesarias sobre datos históricos que ya han sido procesados.

In [51]:
if len(new_files) == 0:

    print(
        "\nNo existen nuevos archivos pendientes."
    )

    print(
        "La capa procesada ya está actualizada."
    )

else:

    print(
        f"\nSe procesarán "
        f"{len(new_files)} archivos nuevos."
    )


No existen nuevos archivos pendientes.
La capa procesada ya está actualizada.


### 20.4 Validación previa de nuevos archivos

Solo los archivos identificados en `new_files` son inspeccionados.

La finalidad de esta etapa es actuar como **barrera de calidad antes de escribir datos**
en la capa procesada. Un archivo con cambios de esquema, fechas inválidas o valores
críticos no debe incorporarse automáticamente al histórico.


In [52]:
# EXPECTED_RAW_COLUMNS se importa desde src.preprocessing.
# La validación compara cada nueva fuente contra el mismo contrato
# utilizado por normalize_chunk(), evitando divergencias de esquema.

validated_new_files = []
rejected_new_files = []


for item in new_files:

    csv_file = item["path"]
    file_hash = item["sha256"]

    print(f"\nValidando: {csv_file}")

    validation_result = {
        "file": csv_file.name,
        "folder": csv_file.parent.name,
        "path": str(csv_file),
        "sha256": file_hash,
        "rows": 0,
        "valid_dates": True,
        "valid_schema": True,
        "valid_binary_values": True,
        "valid_ranges": True,
        "status": "VALID"
    }

    try:

        # ----------------------------------------------------
        # 1. Validación de esquema
        # ----------------------------------------------------

        raw_columns = (
            pd.read_csv(
                csv_file,
                nrows=0
            )
            .columns
            .str.strip()
            .str.upper()
            .tolist()
        )

        missing_columns = sorted(
            set(EXPECTED_RAW_COLUMNS)
            - set(raw_columns)
        )

        extra_columns = sorted(
            set(raw_columns)
            - set(EXPECTED_RAW_COLUMNS)
        )

        if missing_columns or extra_columns:

            validation_result["valid_schema"] = False
            validation_result["status"] = "REJECTED"

            validation_result["error"] = (
                f"Faltantes: {missing_columns}. "
                f"Adicionales: {extra_columns}."
            )

            rejected_new_files.append(
                validation_result
            )

            print("   ❌ Esquema inválido")

            continue

        # ----------------------------------------------------
        # 2. Validación por chunks
        # ----------------------------------------------------

        reader = pd.read_csv(
            csv_file,
            chunksize=CHUNK_SIZE,
            low_memory=False
        )

        for chunk in reader:

            validation_result["rows"] += len(chunk)

            # -----------------------------------------------
            # Normalizar nombres únicamente
            # -----------------------------------------------

            chunk.columns = (
                chunk.columns
                .str.strip()
                .str.upper()
            )

            # -----------------------------------------------
            # Fecha
            # -----------------------------------------------

            dates = pd.to_datetime(
                chunk["FL_DATE"],
                format="%m/%d/%Y %I:%M:%S %p",
                errors="coerce"
            )

            if dates.isna().any():

                validation_result[
                    "valid_dates"
                ] = False

            # -----------------------------------------------
            # Variables binarias
            # -----------------------------------------------

            binary_columns = [
                "DEP_DEL15",
                "ARR_DEL15",
                "CANCELLED",
                "DIVERTED"
            ]

            for column in binary_columns:

                values = pd.to_numeric(
                    chunk[column],
                    errors="coerce"
                )

                invalid_binary = (
                    values.notna()
                    & ~values.isin([0, 1])
                )

                if invalid_binary.any():

                    validation_result[
                        "valid_binary_values"
                    ] = False

            # -----------------------------------------------
            # Rangos físicamente válidos
            # -----------------------------------------------

            numeric_checks = {
                "DISTANCE": 0,
                "AIR_TIME": 0,
                "TAXI_OUT": 0,
                "TAXI_IN": 0
            }

            for column, minimum in numeric_checks.items():

                values = pd.to_numeric(
                    chunk[column],
                    errors="coerce"
                )

                if (values.dropna() < minimum).any():

                    validation_result[
                        "valid_ranges"
                    ] = False

        # ----------------------------------------------------
        # 3. Resultado general
        # ----------------------------------------------------

        if not all([
            validation_result["valid_dates"],
            validation_result["valid_schema"],
            validation_result["valid_binary_values"],
            validation_result["valid_ranges"]
        ]):

            validation_result["status"] = "REJECTED"

            rejected_new_files.append(
                validation_result
            )

            print("   ❌ Archivo rechazado")

        else:

            validated_new_files.append({
                "path": csv_file,
                "sha256": file_hash,
                "rows": validation_result["rows"]
            })

            print(
                f"   ✓ Archivo válido "
                f"({validation_result['rows']:,} filas)"
            )

    except Exception as error:

        validation_result["status"] = "ERROR"
        validation_result["error"] = str(error)

        rejected_new_files.append(
            validation_result
        )

        print(f"   ❌ Error: {error}")


#### 20.4.1 Resumen de la validación de entrada

Se resume el resultado de la validación previa.

Únicamente los archivos clasificados como válidos serán utilizados en el procesamiento incremental.

Los archivos rechazados permanecerán en la capa `raw`, pero no serán incorporados a la capa procesada hasta que se determine la causa de la incidencia.

In [53]:
print("\n" + "=" * 60)
print("VALIDACIÓN DE NUEVOS ARCHIVOS")
print("=" * 60)

print(
    f"Nuevos detectados : "
    f"{len(new_files):,}"
)

print(
    f"Válidos           : "
    f"{len(validated_new_files):,}"
)

print(
    f"Rechazados        : "
    f"{len(rejected_new_files):,}"
)


rejected_new_files_df = pd.DataFrame(
    rejected_new_files
)

if not rejected_new_files_df.empty:
    display(rejected_new_files_df)


VALIDACIÓN DE NUEVOS ARCHIVOS
Nuevos detectados : 0
Válidos           : 0
Rechazados        : 0


### 20.5 Procesamiento incremental

Solo los elementos de `validated_new_files` llegan a este punto.

La lógica reutiliza `normalize_chunk()` y `clean_chunk()` para garantizar que los datos
nuevos reciban exactamente el mismo tratamiento que el histórico usado en análisis y
entrenamiento. Después se escriben en las mismas particiones YEAR/MONTH.


In [54]:
incremental_results = []
incremental_errors = []


for file_number, item in enumerate(
    validated_new_files,
    start=1
):

    csv_file = item["path"]
    file_hash = item["sha256"]

    print(
        f"[{file_number:02d}/"
        f"{len(validated_new_files)}] "
        f"{csv_file.parent.name}"
    )

    file_rows = 0
    file_min_date = None
    file_max_date = None
    parquet_files_created = []

    try:

        reader = pd.read_csv(
            csv_file,
            chunksize=CHUNK_SIZE,
            low_memory=False
        )

        for chunk_number, chunk in enumerate(
            reader,
            start=1
        ):

            # -----------------------------------------------
            # Normalización
            # -----------------------------------------------

            chunk = normalize_chunk(chunk)

            # -----------------------------------------------
            # Limpieza
            # -----------------------------------------------

            chunk = clean_chunk(chunk)

            # -----------------------------------------------
            # Estadísticas del archivo
            # -----------------------------------------------

            file_rows += len(chunk)

            current_min = chunk["FL_DATE"].min()
            current_max = chunk["FL_DATE"].max()

            if pd.notna(current_min):

                if (
                    file_min_date is None
                    or current_min < file_min_date
                ):
                    file_min_date = current_min

            if pd.notna(current_max):

                if (
                    file_max_date is None
                    or current_max > file_max_date
                ):
                    file_max_date = current_max

            # -----------------------------------------------
            # Particionado YEAR / MONTH
            # -----------------------------------------------

            grouped = chunk.groupby(
                ["YEAR", "MONTH"],
                dropna=False
            )

            for (year, month), group_df in grouped:

                year_dir = (
                    PROCESSED_FLIGHTS_DIR
                    / f"YEAR={int(year)}"
                )

                month_dir = (
                    year_dir
                    / f"MONTH={int(month):02d}"
                )

                month_dir.mkdir(
                    parents=True,
                    exist_ok=True
                )

                output_file = (
                    month_dir
                    / (
                        f"{csv_file.parent.name}_"
                        f"chunk_{chunk_number:05d}.parquet"
                    )
                )

                group_df.to_parquet(
                    output_file,
                    engine="pyarrow",
                    compression="snappy",
                    index=False
                )

                parquet_files_created.append(
                    str(output_file)
                )

        # ----------------------------------------------------
        # Registrar ejecución correcta
        # ----------------------------------------------------

        incremental_results.append({
            "source_file": csv_file.name,
            "source_folder": csv_file.parent.name,
            "relative_path": str(
                csv_file.relative_to(
                    PROJECT_ROOT
                )
            ),
            "file_size_bytes": (
                csv_file.stat().st_size
            ),
            "sha256": file_hash,
            "rows": file_rows,
            "min_date": file_min_date,
            "max_date": file_max_date,
            "processed_at": (
                datetime.now().isoformat(
                    timespec="seconds"
                )
            ),
            "status": "PROCESSED"
        })

        print(
            f"   ✓ {file_rows:,} registros procesados"
        )

    except Exception as error:

        incremental_errors.append({
            "file": csv_file.name,
            "path": str(csv_file),
            "sha256": file_hash,
            "error": str(error)
        })

        print(f"   ❌ ERROR: {error}")


In [55]:
incremental_rows = sum(
    item["rows"]
    for item in incremental_results
)

print("\n" + "=" * 60)
print("PROCESAMIENTO INCREMENTAL")
print("=" * 60)

print(
    f"Archivos procesados : "
    f"{len(incremental_results):,}"
)

print(
    f"Registros añadidos  : "
    f"{incremental_rows:,}"
)

print(
    f"Errores              : "
    f"{len(incremental_errors):,}"
)


PROCESAMIENTO INCREMENTAL
Archivos procesados : 0
Registros añadidos  : 0
Errores              : 0


### 20.6 Actualización del manifest

El manifest se actualiza **después** de que el procesamiento termine correctamente.

Este orden es deliberado: evita marcar como `PROCESSED` una fuente cuya escritura en
Parquet haya fallado. Por tanto, el manifest representa únicamente datos confirmados en
la capa procesada.


In [56]:
if incremental_results:

    new_manifest_records = pd.DataFrame(
        incremental_results
    )

    ingestion_manifest = pd.concat(
        [
            ingestion_manifest,
            new_manifest_records
        ],
        ignore_index=True
    )

    ingestion_manifest.to_csv(
        MANIFEST_FILE,
        index=False
    )

    print(
        f"Manifest actualizado: "
        f"{len(ingestion_manifest):,} archivos registrados."
    )

else:

    print(
        "No existen nuevos archivos procesados. "
        "El manifest no requiere actualización."
    )

No existen nuevos archivos procesados. El manifest no requiere actualización.


### 20.7 Actualización de metadatos

Los metadatos describen el estado global actual de la capa Parquet. Se recalculan a
partir de los archivos existentes para que reflejen tanto la carga histórica como las
actualizaciones incrementales posteriores.


In [57]:
import pyarrow.parquet as pq


parquet_files_current = sorted(
    PROCESSED_FLIGHTS_DIR.rglob(
        "*.parquet"
    )
)


total_rows_current = 0
min_date_current = None
max_date_current = None


for parquet_file in parquet_files_current:

    # --------------------------------------------------------
    # Número de filas desde metadatos
    # --------------------------------------------------------

    metadata = pq.ParquetFile(
        parquet_file
    ).metadata

    total_rows_current += (
        metadata.num_rows
    )

    # --------------------------------------------------------
    # Fechas
    # --------------------------------------------------------

    dates = pd.read_parquet(
        parquet_file,
        columns=["FL_DATE"]
    )["FL_DATE"]

    current_min = dates.min()
    current_max = dates.max()

    if pd.notna(current_min):

        if (
            min_date_current is None
            or current_min < min_date_current
        ):
            min_date_current = current_min

    if pd.notna(current_max):

        if (
            max_date_current is None
            or current_max > max_date_current
        ):
            max_date_current = current_max


dataset_metadata = pd.DataFrame([
    {
        "raw_files_registered":
            len(ingestion_manifest),

        "parquet_files":
            len(parquet_files_current),

        "records":
            total_rows_current,

        "variables":
            len(FINAL_COLUMN_ORDER),

        "min_date":
            min_date_current,

        "max_date":
            max_date_current,

        "last_update":
            datetime.now().isoformat(
                timespec="seconds"
            ),

        "incremental_files_processed":
            len(incremental_results),

        "incremental_rows_added":
            incremental_rows,

        "incremental_errors":
            len(incremental_errors)
    }
])


dataset_metadata.to_csv(
    METADATA_FILE,
    index=False
)

display(dataset_metadata)

,raw_files_registered,parquet_files,records,variables,min_date,max_date,last_update,incremental_files_processed,incremental_rows_added,incremental_errors
0,53,354,32761129,45,2022-01-01,2026-05-31,2026-08-12T07:41:33,0,0,0


## 21. Validación y cierre

### 21.1 Validaciones finales automáticas

Las aserciones de este bloque funcionan como controles de integridad del pipeline.
Su objetivo no es transformar datos, sino fallar explícitamente si el manifest, el
esquema o la capa procesada dejan de cumplir las condiciones definidas.


In [58]:
# ------------------------------------------------------------
# 1. Existencia de archivos de control
# ------------------------------------------------------------

assert MANIFEST_FILE.exists(), (
    "No existe ingestion_manifest.csv."
)

assert METADATA_FILE.exists(), (
    "No existe dataset_metadata.csv."
)

# ------------------------------------------------------------
# 2. Validar manifest
# ------------------------------------------------------------

manifest_validation = pd.read_csv(
    MANIFEST_FILE
)

assert not manifest_validation.empty, (
    "El manifest está vacío."
)

assert manifest_validation["sha256"].notna().all(), (
    "Existen registros del manifest sin SHA-256."
)

assert (
    manifest_validation["sha256"].duplicated().sum()
    == 0
), (
    "Existen hashes duplicados en el manifest."
)

assert (
    manifest_validation["status"]
    .eq("PROCESSED")
    .all()
), (
    "Existen archivos en el manifest que no están "
    "marcados como PROCESSED."
)

# ------------------------------------------------------------
# 3. Validar capa Parquet
# ------------------------------------------------------------

assert len(parquet_files_current) > 0, (
    "No existen archivos Parquet procesados."
)

assert total_rows_current > 0, (
    "La capa procesada no contiene registros."
)

# ------------------------------------------------------------
# 4. Validar número de variables
# ------------------------------------------------------------

assert len(FINAL_COLUMN_ORDER) == 45, (
    "El esquema analítico no contiene 45 variables."
)

sample_final_df = pd.read_parquet(
    parquet_files_current[0]
)

assert list(sample_final_df.columns) == FINAL_COLUMN_ORDER, (
    "El esquema Parquet no coincide con "
    "FINAL_COLUMN_ORDER."
)

# ------------------------------------------------------------
# 5. Validar ejecución incremental
# ------------------------------------------------------------

assert len(incremental_errors) == 0, (
    "Existen errores en la última ejecución incremental."
)

# ------------------------------------------------------------
# 6. Resultado
# ------------------------------------------------------------

print("✓ Manifest validado.")
print("✓ No existen hashes duplicados.")
print("✓ Capa Parquet validada.")
print("✓ Esquema de 45 variables validado.")
print("✓ No existen errores incrementales.")
print("✓ Pipeline preparado para futuras actualizaciones.")

✓ Manifest validado.
✓ No existen hashes duplicados.
✓ Capa Parquet validada.
✓ Esquema de 45 variables validado.
✓ No existen errores incrementales.
✓ Pipeline preparado para futuras actualizaciones.


### 21.2 Validación del mecanismo de no duplicación

El mecanismo incremental utiliza el hash SHA-256 de cada archivo como identificador de contenido.

Se comprueba que todos los archivos actualmente disponibles en la capa `raw` que ya han sido procesados se encuentren registrados en el manifest.

Si no existen nuevas fuentes, el número de archivos pendientes debe ser igual a cero.

Esta validación demuestra que una segunda ejecución del pipeline no generará nuevamente los registros históricos existentes.

In [59]:
current_raw_files = sorted(
    FLIGHTS_DIR.rglob("*.csv")
)

manifest_hashes = set(
    manifest_validation["sha256"]
)

unregistered_files = []


for csv_file in current_raw_files:

    file_hash = calculate_sha256(
        csv_file
    )

    if file_hash not in manifest_hashes:

        unregistered_files.append(
            csv_file
        )


print(
    f"Archivos actuales en raw : "
    f"{len(current_raw_files):,}"
)

print(
    f"Archivos en manifest     : "
    f"{len(manifest_validation):,}"
)

print(
    f"Archivos no registrados  : "
    f"{len(unregistered_files):,}"
)


if len(unregistered_files) == 0:

    print(
        "\n✓ Todos los archivos actuales están "
        "registrados."
    )

    print(
        "✓ Una nueva ejecución no reprocesará "
        "el histórico."
    )

else:

    print(
        "\nExisten archivos nuevos pendientes:"
    )

    for file in unregistered_files:
        print(file)

Archivos actuales en raw : 53
Archivos en manifest     : 53
Archivos no registrados  : 0

✓ Todos los archivos actuales están registrados.
✓ Una nueva ejecución no reprocesará el histórico.


### 21.3 Resumen técnico final

Se genera un resumen dinámico del estado final del proceso de ingesta.

Los valores se obtienen directamente de la capa procesada y de los archivos de control, evitando introducir manualmente resultados que podrían quedar desactualizados tras futuras incorporaciones de datos.

In [60]:
final_ingestion_summary = pd.DataFrame([
    {
        "metrica": "Archivos fuente registrados",
        "valor": len(manifest_validation)
    },
    {
        "metrica": "Archivos Parquet",
        "valor": len(parquet_files_current)
    },
    {
        "metrica": "Registros procesados",
        "valor": total_rows_current
    },
    {
        "metrica": "Variables",
        "valor": len(FINAL_COLUMN_ORDER)
    },
    {
        "metrica": "Fecha inicial",
        "valor": min_date_current
    },
    {
        "metrica": "Fecha final",
        "valor": max_date_current
    },
    {
        "metrica": "Archivos nuevos procesados "
                   "en última ejecución",
        "valor": len(incremental_results)
    },
    {
        "metrica": "Registros añadidos "
                   "en última ejecución",
        "valor": incremental_rows
    },
    {
        "metrica": "Errores incrementales",
        "valor": len(incremental_errors)
    },
    {
        "metrica": "Archivos pendientes",
        "valor": len(unregistered_files)
    }
])

display(final_ingestion_summary)

,metrica,valor
0,Archivos fuente registrados,53
1,Archivos Parquet,354
2,Registros procesados,32761129
3,Variables,45
4,Fecha inicial,2022-01-01 00:00:00
5,Fecha final,2026-05-31 00:00:00
6,Archivos nuevos procesados en última ejecución,0
7,Registros añadidos en última ejecución,0
8,Errores incrementales,0
9,Archivos pendientes,0


### 21.4 Conclusiones de la fase de ingesta

La fase de ingesta ha permitido construir una capa de datos históricos estructurada, validada y preparada para las posteriores etapas analíticas.

Los archivos originales descargados del Bureau of Transportation Statistics se mantienen sin modificaciones dentro de la capa `raw`, garantizando la trazabilidad de la fuente.

El pipeline desarrollado incorpora procesos diferenciados de normalización estructural y limpieza. Las transformaciones se aplican mediante funciones reutilizables, lo que garantiza que todos los archivos sean tratados bajo las mismas reglas.

El análisis de calidad permitió identificar valores faltantes estructurales, comprobar la ausencia de registros duplicados según la clave operacional definida, validar los principales rangos y relaciones lógicas entre variables y analizar valores extremos antes de tomar decisiones de limpieza.

Como resultado del procesamiento histórico se generó una capa analítica en formato Parquet, particionada temporalmente por año y mes. Esta capa constituye la fuente oficial de datos para los siguientes notebooks del proyecto.

Adicionalmente, se implementó un mecanismo de ingesta incremental basado en un `ingestion_manifest.csv`. El manifest registra los archivos ya procesados mediante su hash SHA-256 y permite detectar nuevas fuentes antes de realizar cualquier transformación.

De esta forma, futuras actualizaciones del histórico podrán incorporarse sin necesidad de reprocesar todos los datos existentes. Únicamente los archivos nuevos que superen las validaciones de entrada serán normalizados, limpiados y añadidos a la capa procesada.

El pipeline queda por tanto preparado tanto para la construcción inicial del histórico como para su mantenimiento incremental.

Las transformaciones específicas destinadas al modelado predictivo no se realizan en esta fase. La selección de variables, prevención de *data leakage*, codificación de variables categóricas, escalado y demás técnicas de ingeniería de características se abordarán posteriormente de acuerdo con los objetivos de cada modelo.

Con ello se considera finalizado el notebook `01_data_ingestion.ipynb`.

### 21.5 Punto de salida

La fuente oficial para las siguientes fases del proyecto se encuentra en:

`data/processed/flights/`

Los siguientes análisis no utilizarán directamente los CSV originales, sino la capa procesada en formato Parquet.

El siguiente notebook será:

`02_eda.ipynb`

y estará dedicado al **Análisis Exploratorio de Datos (EDA)**, correspondiente al capítulo 5 del índice del TFM.

El EDA permitirá estudiar, entre otros aspectos:

- Distribución general de los vuelos.
- Evolución temporal.
- Distribución de los retrasos.
- Comportamiento por aerolínea.
- Comportamiento por aeropuerto.
- Relación entre origen y destino.
- Cancelaciones y desvíos.
- Causas de retraso.
- Relaciones entre variables numéricas y categóricas.

Los resultados obtenidos durante esta fase servirán posteriormente como base para la ingeniería de características, el análisis de componentes principales y la modelización predictiva.